#Recurrence Steps (t) = 4

##ReLU


###Loss: Focal Tversky + Dice

In [ ]:
!pip install medpy --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 48.3 MB/s eta 0:00:00


In [ ]:
import zipfile, os, cv2, numpy as np, math
import tensorflow as tf
import albumentations as A
from sklearn.model_selection import KFold
from medpy.metric import binary

#Unzip Data
with zipfile.ZipFile('/content/stage1_train.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/stage1_train')
print("✅ Unzipped stage1_train.zip successfully!")

#Load Data
def load_dsb2018_data(dataset_dir, image_size=(256,256)):
    images, masks = [], []
    for folder in sorted(os.listdir(dataset_dir)):
        img_path = os.path.join(dataset_dir, folder, 'images', folder + '.png')
        mask_dir = os.path.join(dataset_dir, folder, 'masks')
        if not os.path.exists(img_path) or not os.path.exists(mask_dir):
            continue
        image = cv2.imread(img_path)
        image = cv2.resize(image, image_size).astype(np.float32)/255.0
        mask = np.zeros(image_size, dtype=np.uint8)
        for m in os.listdir(mask_dir):
            msk = cv2.imread(os.path.join(mask_dir, m), cv2.IMREAD_GRAYSCALE)
            msk = cv2.resize(msk, image_size)
            mask = np.maximum(mask, msk)
        mask = (mask>0).astype(np.float32)
        images.append(image)
        masks.append(np.expand_dims(mask, axis=-1))
    return np.array(images), np.array(masks)

X, y = load_dsb2018_data('/content/stage1_train', image_size=(256,256))

#Augmentation
transform = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2), A.GaussianBlur(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
    A.GridDistortion(p=0.2), A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.2)
])

def augment(image, mask):
    augmented = transform(image=image, mask=mask)
    return augmented['image'], augmented['mask']

def tf_augment(img, mask):
    img, mask = tf.numpy_function(augment, [img, mask], [tf.float32, tf.float32])
    img.set_shape([256,256,3])
    mask.set_shape([256,256,1])
    return img, mask

#Loss Function
def tversky(y_true, y_pred, alpha=0.5, beta=0.5):
    smooth = 1e-6
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    tp = tf.reduce_sum(y_true * y_pred)
    fn = tf.reduce_sum(y_true * (1 - y_pred))
    fp = tf.reduce_sum((1 - y_true) * y_pred)
    return (tp + smooth) / (tp + alpha*fn + beta*fp + smooth)

def focal_tversky_loss(y_true, y_pred, gamma=1.33):
    tv = tversky(y_true, y_pred)
    return tf.pow((1 - tv), gamma)

def dice_loss(y_true, y_pred):
    smooth=1e-6
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1 - (2.*intersection + smooth)/(tf.reduce_sum(y_true_f)+tf.reduce_sum(y_pred_f)+smooth)

def hybrid_loss(y_true, y_pred):
    return 0.7*focal_tversky_loss(y_true, y_pred) + 0.3*dice_loss(y_true, y_pred)

#R2U-Net Model
class RecurrentConv(tf.keras.layers.Layer):
    def __init__(self, filters, t=2):
        super().__init__()
        self.filters = filters
        self.t = t
        self.activation = tf.keras.layers.Activation('relu')
        self.convs = [tf.keras.layers.Conv2D(filters, 3, padding='same') for _ in range(t)]
        self.bns = [tf.keras.layers.BatchNormalization() for _ in range(t)]
    def call(self, x):
        h = 0
        for i in range(self.t):
            h = self.activation(self.bns[i](self.convs[i](x + h))) if i>0 else self.activation(self.bns[i](self.convs[i](x)))
        return h

class RRU(tf.keras.layers.Layer):
    def __init__(self, filters, t=2):
        super().__init__()
        self.projection = tf.keras.layers.Conv2D(filters, 1, padding='same')
        self.rcl = RecurrentConv(filters, t)
    def call(self, x):
        x_proj = self.projection(x)
        return x_proj + self.rcl(x_proj)

def build_r2unet(input_shape=(256,256,3), num_classes=1, t=4):
    inputs = tf.keras.Input(shape=input_shape)
    #Encoder
    e1 = RRU(32, t)(inputs); p1=tf.keras.layers.MaxPooling2D()(e1)
    e2 = RRU(64, t)(p1); p2=tf.keras.layers.MaxPooling2D()(e2)
    e3 = RRU(128, t)(p2); p3=tf.keras.layers.MaxPooling2D()(e3)
    e4 = RRU(256, t)(p3); p4=tf.keras.layers.MaxPooling2D()(e4)
    # Bottleneck
    b = RRU(512, t)(p4)
    # Decoder
    u1 = tf.keras.layers.UpSampling2D()(b); u1=tf.keras.layers.Concatenate()([u1,e4]); d1 = RRU(256,t)(u1)
    u2 = tf.keras.layers.UpSampling2D()(d1); u2=tf.keras.layers.Concatenate()([u2,e3]); d2 = RRU(128,t)(u2)
    u3 = tf.keras.layers.UpSampling2D()(d2); u3=tf.keras.layers.Concatenate()([u3,e2]); d3 = RRU(64,t)(u3)
    u4 = tf.keras.layers.UpSampling2D()(d3); u4=tf.keras.layers.Concatenate()([u4,e1]); d4 = RRU(32,t)(u4)
    outputs = tf.keras.layers.Conv2D(num_classes,1,activation='sigmoid')(d4)
    return tf.keras.Model(inputs, outputs)

#KFold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
all_fold_dice_scores = []

for train_idx, val_idx in kfold.split(X):
    print(f"========== Fold {fold} ==========")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    train_dataset = train_dataset.map(tf_augment, num_parallel_calls=tf.data.AUTOTUNE)
    train_dataset = train_dataset.shuffle(128).batch(16).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    val_dataset = val_dataset.batch(16).prefetch(tf.data.AUTOTUNE)

    lr_schedule = tf.keras.callbacks.LearningRateScheduler(lambda epoch: 1e-4*(1+math.cos(math.pi*epoch/150))/2)
    early_stop = tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)
    checkpoint = tf.keras.callbacks.ModelCheckpoint(f"best_r2unet_fold{fold}.h5", save_best_only=True)

    model = build_r2unet(input_shape=(256,256,3), t=4)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=hybrid_loss, metrics=['accuracy'])
    model.fit(train_dataset, validation_data=val_dataset, epochs=150, callbacks=[lr_schedule, early_stop, checkpoint], verbose=1)

    #Fold Evaluation
    dice_scores_fold = []
    preds = model.predict(X_val, batch_size=16, verbose=0)
    preds_bin = (preds>0.5).astype(np.uint8)
    for pb, gt in zip(preds_bin, y_val):
        if np.sum(pb)>0 and np.sum(gt)>0:
            dice_scores_fold.append(binary.dc(pb.squeeze(), gt.squeeze()))
    mean_dice_fold = np.mean(dice_scores_fold) if len(dice_scores_fold)>0 else 0
    print(f"✅ Fold {fold} Dice: {mean_dice_fold:.4f}")
    all_fold_dice_scores.append(mean_dice_fold)
    fold += 1

#Average Dice
print(f"✅ Average Dice across all folds: {np.mean(all_fold_dice_scores):.4f}")

✅ Unzipped stage1_train.zip successfully!


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipython-input-700507915.py:39: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.GridDistortion(p=0.2), A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.2)


========== Fold 1 ==========
Epoch 1/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7028 - loss: 0.4254   

34/34 ━━━━━━━━━━━━━━━━━━━━ 106s 2s/step - accuracy: 0.7067 - loss: 0.4219 - val_accuracy: 0.8720 - val_loss: 0.9771 - learning_rate: 1.0000e-04
Epoch 2/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 675ms/step - accuracy: 0.9344 - loss: 0.1679 - val_accuracy: 0.8720 - val_loss: 0.9918 - learning_rate: 9.9989e-05
Epoch 3/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 700ms/step - accuracy: 0.9500 - loss: 0.1260 - val_accuracy: 0.8720 - val_loss: 0.9980 - learning_rate: 9.9956e-05
Epoch 4/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 675ms/step - accuracy: 0.9501 - loss: 0.1276 - val_accuracy: 0.8720 - val_loss: 0.9997 - learning_rate: 9.9901e-05
Epoch 5/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 676ms/step - accuracy: 0.9580 - loss: 0.1048 - val_accuracy: 0.8720 - val_loss: 0.9942 - learning_rate: 9.9825e-05
Epoch 6/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 639ms/step - accuracy: 0.9573 - loss: 0.1012

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 711ms/step - accuracy: 0.9574 - loss: 0.1013 - val_accuracy: 0.8735 - val_loss: 0.9473 - learning_rate: 9.9726e-05
Epoch 7/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 637ms/step - accuracy: 0.9585 - loss: 0.1024

...

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 707ms/step - accuracy: 0.9746 - loss: 0.0611 - val_accuracy: 0.9735 - val_loss: 0.0529 - learning_rate: 2.2330e-05
Epoch 105/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 683ms/step - accuracy: 0.9745 - loss: 0.0616 - val_accuracy: 0.9727 - val_loss: 0.0541 - learning_rate: 2.1464e-05
Epoch 106/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 685ms/step - accuracy: 0.9738 - loss: 0.0625 - val_accuracy: 0.9727 - val_loss: 0.0547 - learning_rate: 2.0611e-05
Epoch 107/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 685ms/step - accuracy: 0.9726 - loss: 0.0670 - val_accuracy: 0.9736 - val_loss: 0.0533 - learning_rate: 1.9770e-05
Epoch 108/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 684ms/step - accuracy: 0.9748 - loss: 0.0585 - val_accuracy: 0.9727 - val_loss: 0.0545 - learning_rate: 1.8943e-05
Epoch 109/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 684ms/step - accuracy: 0.9721 - loss: 0.0702 - val_accuracy: 0.9732 - val_loss: 0.0533 - learning_rate: 1.8129e-05
Epoch 110/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 682ms/ste

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 710ms/step - accuracy: 0.9766 - loss: 0.0540 - val_accuracy: 0.9736 - val_loss: 0.0528 - learning_rate: 1.4276e-05
Epoch 115/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 685ms/step - accuracy: 0.9743 - loss: 0.0604 - val_accuracy: 0.9733 - val_loss: 0.0530 - learning_rate: 1.3552e-05
Epoch 116/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 638ms/step - accuracy: 0.9750 - loss: 0.0585

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 709ms/step - accuracy: 0.9750 - loss: 0.0585 - val_accuracy: 0.9739 - val_loss: 0.0523 - learning_rate: 1.2843e-05
Epoch 117/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 636ms/step - accuracy: 0.9772 - loss: 0.0547

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 706ms/step - accuracy: 0.9772 - loss: 0.0547 - val_accuracy: 0.9739 - val_loss: 0.0522 - learning_rate: 1.2150e-05
Epoch 118/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 681ms/step - accuracy: 0.9743 - loss: 0.0631 - val_accuracy: 0.9739 - val_loss: 0.0523 - learning_rate: 1.1474e-05
Epoch 119/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 637ms/step - accuracy: 0.9758 - loss: 0.0601

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 708ms/step - accuracy: 0.9757 - loss: 0.0601 - val_accuracy: 0.9739 - val_loss: 0.0519 - learning_rate: 1.0815e-05
Epoch 120/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 684ms/step - accuracy: 0.9745 - loss: 0.0615 - val_accuracy: 0.9736 - val_loss: 0.0524 - learning_rate: 1.0174e-05
Epoch 121/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 685ms/step - accuracy: 0.9764 - loss: 0.0581 - val_accuracy: 0.9734 - val_loss: 0.0530 - learning_rate: 9.5492e-06
Epoch 122/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 684ms/step - accuracy: 0.9746 - loss: 0.0633 - val_accuracy: 0.9735 - val_loss: 0.0527 - learning_rate: 8.9425e-06
Epoch 123/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 684ms/step - accuracy: 0.9737 - loss: 0.0634 - val_accuracy: 0.9738 - val_loss: 0.0524 - learning_rate: 8.3539e-06
Epoch 124/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 685ms/step - accuracy: 0.9730 - loss: 0.0674 - val_accuracy: 0.9736 - val_loss: 0.0523 - learning_rate: 7.7836e-06
Epoch 125/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 684ms/ste

###Loss: Focal Tversky + BCE + Dice

In [ ]:
import zipfile, os, cv2, numpy as np, math
import tensorflow as tf
import albumentations as A
from sklearn.model_selection import KFold
from medpy.metric import binary

#Unzip Data
with zipfile.ZipFile('/content/stage1_train.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/stage1_train')
print("✅ Unzipped stage1_train.zip successfully!")

#Load Data
def load_dsb2018_data(dataset_dir, image_size=(256,256)):
    images, masks = [], []
    for folder in sorted(os.listdir(dataset_dir)):
        img_path = os.path.join(dataset_dir, folder, 'images', folder + '.png')
        mask_dir = os.path.join(dataset_dir, folder, 'masks')
        if not os.path.exists(img_path) or not os.path.exists(mask_dir):
            continue
        image = cv2.imread(img_path)
        image = cv2.resize(image, image_size).astype(np.float32)/255.0
        mask = np.zeros(image_size, dtype=np.uint8)
        for m in os.listdir(mask_dir):
            msk = cv2.imread(os.path.join(mask_dir, m), cv2.IMREAD_GRAYSCALE)
            msk = cv2.resize(msk, image_size)
            mask = np.maximum(mask, msk)
        mask = (mask>0).astype(np.float32)
        images.append(image)
        masks.append(np.expand_dims(mask, axis=-1))
    return np.array(images), np.array(masks)

X, y = load_dsb2018_data('/content/stage1_train', image_size=(256,256))

#Augmentation
transform = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2), A.GaussianBlur(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
    A.GridDistortion(p=0.2), A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.2)
])

def augment(image, mask):
    augmented = transform(image=image, mask=mask)
    return augmented['image'], augmented['mask']

def tf_augment(img, mask):
    img, mask = tf.numpy_function(augment, [img, mask], [tf.float32, tf.float32])
    img.set_shape([256,256,3])
    mask.set_shape([256,256,1])
    return img, mask

#Loss Function
def tversky(y_true, y_pred, alpha=0.5, beta=0.5):
    smooth = 1e-6
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    tp = tf.reduce_sum(y_true * y_pred)
    fn = tf.reduce_sum(y_true * (1 - y_pred))
    fp = tf.reduce_sum((1 - y_true) * y_pred)
    return (tp + smooth) / (tp + alpha*fn + beta*fp + smooth)

def focal_tversky_loss(y_true, y_pred, gamma=1.33):
    tv = tversky(y_true, y_pred)
    return tf.pow((1 - tv), gamma)

def dice_loss(y_true, y_pred):
    smooth=1e-6
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1 - (2.*intersection + smooth)/(tf.reduce_sum(y_true_f)+tf.reduce_sum(y_pred_f)+smooth)

def hybrid_loss(y_true, y_pred):
    return 0.7*focal_tversky_loss(y_true, y_pred) + 0.3*dice_loss(y_true, y_pred)

#R2U-Net Model
class RecurrentConv(tf.keras.layers.Layer):
    def __init__(self, filters, t=2):
        super().__init__()
        self.filters = filters
        self.t = t
        self.activation = tf.keras.layers.Activation('relu')
        self.convs = [tf.keras.layers.Conv2D(filters, 3, padding='same') for _ in range(t)]
        self.bns = [tf.keras.layers.BatchNormalization() for _ in range(t)]
    def call(self, x):
        h = 0
        for i in range(self.t):
            h = self.activation(self.bns[i](self.convs[i](x + h))) if i>0 else self.activation(self.bns[i](self.convs[i](x)))
        return h

class RRU(tf.keras.layers.Layer):
    def __init__(self, filters, t=2):
        super().__init__()
        self.projection = tf.keras.layers.Conv2D(filters, 1, padding='same')
        self.rcl = RecurrentConv(filters, t)
    def call(self, x):
        x_proj = self.projection(x)
        return x_proj + self.rcl(x_proj)

def build_r2unet(input_shape=(256,256,3), num_classes=1, t=4):
    inputs = tf.keras.Input(shape=input_shape)
    #Encoder
    e1 = RRU(32, t)(inputs); p1=tf.keras.layers.MaxPooling2D()(e1)
    e2 = RRU(64, t)(p1); p2=tf.keras.layers.MaxPooling2D()(e2)
    e3 = RRU(128, t)(p2); p3=tf.keras.layers.MaxPooling2D()(e3)
    e4 = RRU(256, t)(p3); p4=tf.keras.layers.MaxPooling2D()(e4)
    # Bottleneck
    b = RRU(512, t)(p4)
    # Decoder
    u1 = tf.keras.layers.UpSampling2D()(b); u1=tf.keras.layers.Concatenate()([u1,e4]); d1 = RRU(256,t)(u1)
    u2 = tf.keras.layers.UpSampling2D()(d1); u2=tf.keras.layers.Concatenate()([u2,e3]); d2 = RRU(128,t)(u2)
    u3 = tf.keras.layers.UpSampling2D()(d2); u3=tf.keras.layers.Concatenate()([u3,e2]); d3 = RRU(64,t)(u3)
    u4 = tf.keras.layers.UpSampling2D()(d3); u4=tf.keras.layers.Concatenate()([u4,e1]); d4 = RRU(32,t)(u4)
    outputs = tf.keras.layers.Conv2D(num_classes,1,activation='sigmoid')(d4)
    return tf.keras.Model(inputs, outputs)

#KFold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
all_fold_dice_scores = []

for train_idx, val_idx in kfold.split(X):
    print(f"========== Fold {fold} ==========")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    train_dataset = train_dataset.map(tf_augment, num_parallel_calls=tf.data.AUTOTUNE)
    train_dataset = train_dataset.shuffle(128).batch(16).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    val_dataset = val_dataset.batch(16).prefetch(tf.data.AUTOTUNE)

    lr_schedule = tf.keras.callbacks.LearningRateScheduler(lambda epoch: 1e-4*(1+math.cos(math.pi*epoch/150))/2)
    early_stop = tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)
    checkpoint = tf.keras.callbacks.ModelCheckpoint(f"r2unet_relu_1024_fold{fold}.h5", save_best_only=True)

    model = build_r2unet(input_shape=(256,256,3), t=4)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=hybrid_loss, metrics=['accuracy'])
    model.fit(train_dataset, validation_data=val_dataset, epochs=150, callbacks=[lr_schedule, early_stop, checkpoint], verbose=1)

    #Fold Evaluation
    dice_scores_fold = []
    preds = model.predict(X_val, batch_size=16, verbose=0)
    preds_bin = (preds>0.5).astype(np.uint8)
    for pb, gt in zip(preds_bin, y_val):
        if np.sum(pb)>0 and np.sum(gt)>0:
            dice_scores_fold.append(binary.dc(pb.squeeze(), gt.squeeze()))
    mean_dice_fold = np.mean(dice_scores_fold) if len(dice_scores_fold)>0 else 0
    print(f"✅ Fold {fold} Dice: {mean_dice_fold:.4f}")
    all_fold_dice_scores.append(mean_dice_fold)
    fold += 1

#Average Dice
print(f"✅ Average Dice across all folds: {np.mean(all_fold_dice_scores):.4f}")

✅ Unzipped stage1_train.zip successfully!


/tmp/ipython-input-272861139.py:39: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.GridDistortion(p=0.2), A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.2)


========== Fold 1 ==========
Epoch 1/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7903 - loss: 0.4124   

34/34 ━━━━━━━━━━━━━━━━━━━━ 84s 1s/step - accuracy: 0.7925 - loss: 0.4092 - val_accuracy: 0.8732 - val_loss: 0.8205 - learning_rate: 1.0000e-04
Epoch 2/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 682ms/step - accuracy: 0.9368 - loss: 0.1611 - val_accuracy: 0.8725 - val_loss: 0.9627 - learning_rate: 9.9989e-05
Epoch 3/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 707ms/step - accuracy: 0.9485 - loss: 0.1243 - val_accuracy: 0.8723 - val_loss: 0.9829 - learning_rate: 9.9956e-05
Epoch 4/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 675ms/step - accuracy: 0.9542 - loss: 0.1122 - val_accuracy: 0.8721 - val_loss: 0.9946 - learning_rate: 9.9901e-05
Epoch 5/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 672ms/step - accuracy: 0.9537 - loss: 0.1194 - val_accuracy: 0.8744 - val_loss: 0.9264 - learning_rate: 9.9825e-05
Epoch 6/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 638ms/step - accuracy: 0.9558 - loss: 0.1073

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 710ms/step - accuracy: 0.9558 - loss: 0.1073 - val_accuracy: 0.8988 - val_loss: 0.4813 - learning_rate: 9.9726e-05
Epoch 7/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 684ms/step - accuracy: 0.9548 - loss: 0.1088 - val_accuracy: 0.8947 - val_loss: 0.5370 - learning_rate: 9.9606e-05
Epoch 8/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 637ms/step - accuracy: 0.9630 - loss: 0.0903

...

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 706ms/step - accuracy: 0.9747 - loss: 0.0578 - val_accuracy: 0.9726 - val_loss: 0.0549 - learning_rate: 4.2696e-05
Epoch 84/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 680ms/step - accuracy: 0.9720 - loss: 0.0669 - val_accuracy: 0.9718 - val_loss: 0.0557 - learning_rate: 4.1662e-05
Epoch 85/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 682ms/step - accuracy: 0.9728 - loss: 0.0669 - val_accuracy: 0.9721 - val_loss: 0.0556 - learning_rate: 4.0631e-05
Epoch 86/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 41s 679ms/step - accuracy: 0.9731 - loss: 0.0643 - val_accuracy: 0.9725 - val_loss: 0.0556 - learning_rate: 3.9604e-05
Epoch 87/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 657ms/step - accuracy: 0.9749 - loss: 0.0567

34/34 ━━━━━━━━━━━━━━━━━━━━ 25s 729ms/step - accuracy: 0.9749 - loss: 0.0569 - val_accuracy: 0.9728 - val_loss: 0.0543 - learning_rate: 3.8582e-05
Epoch 88/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 676ms/step - accuracy: 0.9734 - loss: 0.0638 - val_accuracy: 0.9707 - val_loss: 0.0583 - learning_rate: 3.7566e-05
Epoch 89/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 673ms/step - accuracy: 0.9725 - loss: 0.0653 - val_accuracy: 0.9725 - val_loss: 0.0553 - learning_rate: 3.6554e-05
Epoch 90/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 682ms/step - accuracy: 0.9741 - loss: 0.0596 - val_accuracy: 0.9702 - val_loss: 0.0585 - learning_rate: 3.5548e-05
Epoch 91/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 638ms/step - accuracy: 0.9759 - loss: 0.0570

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 709ms/step - accuracy: 0.9759 - loss: 0.0571 - val_accuracy: 0.9733 - val_loss: 0.0535 - learning_rate: 3.4549e-05
Epoch 92/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 680ms/step - accuracy: 0.9737 - loss: 0.0612 - val_accuracy: 0.9714 - val_loss: 0.0565 - learning_rate: 3.3557e-05
Epoch 93/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 678ms/step - accuracy: 0.9707 - loss: 0.0722 - val_accuracy: 0.9712 - val_loss: 0.0567 - learning_rate: 3.2571e-05
Epoch 94/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 677ms/step - accuracy: 0.9741 - loss: 0.0634 - val_accuracy: 0.9724 - val_loss: 0.0550 - learning_rate: 3.1594e-05
Epoch 95/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 678ms/step - accuracy: 0.9722 - loss: 0.0646 - val_accuracy: 0.9721 - val_loss: 0.0551 - learning_rate: 3.0624e-05
Epoch 96/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 680ms/step - accuracy: 0.9742 - loss: 0.0608 - val_accuracy: 0.9726 - val_loss: 0.0543 - learning_rate: 2.9663e-05
Epoch 97/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 681ms/step - ac

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 707ms/step - accuracy: 0.9751 - loss: 0.0581 - val_accuracy: 0.9730 - val_loss: 0.0535 - learning_rate: 2.2330e-05
Epoch 105/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 682ms/step - accuracy: 0.9755 - loss: 0.0554 - val_accuracy: 0.9730 - val_loss: 0.0540 - learning_rate: 2.1464e-05
Epoch 106/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 683ms/step - accuracy: 0.9743 - loss: 0.0618 - val_accuracy: 0.9729 - val_loss: 0.0536 - learning_rate: 2.0611e-05
Epoch 107/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 683ms/step - accuracy: 0.9749 - loss: 0.0623 - val_accuracy: 0.9722 - val_loss: 0.0549 - learning_rate: 1.9770e-05
Epoch 108/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 684ms/step - accuracy: 0.9749 - loss: 0.0606 - val_accuracy: 0.9731 - val_loss: 0.0537 - learning_rate: 1.8943e-05
Epoch 109/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 685ms/step - accuracy: 0.9748 - loss: 0.0568 - val_accuracy: 0.9730 - val_loss: 0.0538 - learning_rate: 1.8129e-05
Epoch 110/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 638ms/step

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 708ms/step - accuracy: 0.9738 - loss: 0.0601 - val_accuracy: 0.9734 - val_loss: 0.0530 - learning_rate: 1.7329e-05
Epoch 111/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 682ms/step - accuracy: 0.9736 - loss: 0.0643 - val_accuracy: 0.9727 - val_loss: 0.0538 - learning_rate: 1.6543e-05
Epoch 112/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 684ms/step - accuracy: 0.9751 - loss: 0.0583 - val_accuracy: 0.9724 - val_loss: 0.0546 - learning_rate: 1.5773e-05
Epoch 113/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 684ms/step - accuracy: 0.9741 - loss: 0.0610 - val_accuracy: 0.9733 - val_loss: 0.0533 - learning_rate: 1.5017e-05
Epoch 114/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 685ms/step - accuracy: 0.9752 - loss: 0.0599 - val_accuracy: 0.9733 - val_loss: 0.0533 - learning_rate: 1.4276e-05
Epoch 115/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 684ms/step - accuracy: 0.9737 - loss: 0.0631 - val_accuracy: 0.9729 - val_loss: 0.0538 - learning_rate: 1.3552e-05
Epoch 116/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 683ms/ste

In [ ]:
from google.colab import files

for fold in range(1, 6):
    files.download(f"r2unet_relu_1024_fold{fold}.h5")

##Leaky ReLU

In [ ]:
import zipfile, os, cv2, numpy as np, math
import tensorflow as tf
import albumentations as A
from sklearn.model_selection import KFold
from medpy.metric import binary

#Unzip Data
with zipfile.ZipFile('/content/stage1_train.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/stage1_train')
print("✅ Unzipped stage1_train.zip successfully!")

#Load Data
def load_dsb2018_data(dataset_dir, image_size=(256,256)):
    images, masks = [], []
    for folder in sorted(os.listdir(dataset_dir)):
        img_path = os.path.join(dataset_dir, folder, 'images', folder + '.png')
        mask_dir = os.path.join(dataset_dir, folder, 'masks')
        if not os.path.exists(img_path) or not os.path.exists(mask_dir):
            continue
        image = cv2.imread(img_path)
        image = cv2.resize(image, image_size).astype(np.float32)/255.0
        mask = np.zeros(image_size, dtype=np.uint8)
        for m in os.listdir(mask_dir):
            msk = cv2.imread(os.path.join(mask_dir, m), cv2.IMREAD_GRAYSCALE)
            msk = cv2.resize(msk, image_size)
            mask = np.maximum(mask, msk)
        mask = (mask>0).astype(np.float32)
        images.append(image)
        masks.append(np.expand_dims(mask, axis=-1))
    return np.array(images), np.array(masks)

X, y = load_dsb2018_data('/content/stage1_train', image_size=(256,256))

#Augmentation
transform = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2), A.GaussianBlur(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
    A.GridDistortion(p=0.2), A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.2)
])

def augment(image, mask):
    augmented = transform(image=image, mask=mask)
    return augmented['image'], augmented['mask']

def tf_augment(img, mask):
    img, mask = tf.numpy_function(augment, [img, mask], [tf.float32, tf.float32])
    img.set_shape([256,256,3])
    mask.set_shape([256,256,1])
    return img, mask

#Loss Function
def tversky(y_true, y_pred, alpha=0.5, beta=0.5):
    smooth = 1e-6
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    tp = tf.reduce_sum(y_true * y_pred)
    fn = tf.reduce_sum(y_true * (1 - y_pred))
    fp = tf.reduce_sum((1 - y_true) * y_pred)
    return (tp + smooth) / (tp + alpha*fn + beta*fp + smooth)

def focal_tversky_loss(y_true, y_pred, gamma=1.33):
    tv = tversky(y_true, y_pred)
    return tf.pow((1 - tv), gamma)

def dice_loss(y_true, y_pred):
    smooth=1e-6
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1 - (2.*intersection + smooth)/(tf.reduce_sum(y_true_f)+tf.reduce_sum(y_pred_f)+smooth)

def hybrid_loss(y_true, y_pred):
    return 0.7*focal_tversky_loss(y_true, y_pred) + 0.3*dice_loss(y_true, y_pred)

#R2U-Net Model
class RecurrentConv(tf.keras.layers.Layer):
    def __init__(self, filters, t=2, alpha = 0.1):
        super().__init__()
        self.filters = filters
        self.t = t
        self.activation = tf.keras.layers.LeakyReLU(alpha=alpha)
        self.convs = [tf.keras.layers.Conv2D(filters, 3, padding='same') for _ in range(t)]
        self.bns = [tf.keras.layers.BatchNormalization() for _ in range(t)]
    def call(self, x):
        h = 0
        for i in range(self.t):
            h = self.activation(self.bns[i](self.convs[i](x + h))) if i>0 else self.activation(self.bns[i](self.convs[i](x)))
        return h

class RRU(tf.keras.layers.Layer):
    def __init__(self, filters, t=2):
        super().__init__()
        self.projection = tf.keras.layers.Conv2D(filters, 1, padding='same')
        self.rcl = RecurrentConv(filters, t)
    def call(self, x):
        x_proj = self.projection(x)
        return x_proj + self.rcl(x_proj)

def build_r2unet(input_shape=(256,256,3), num_classes=1, t=4):
    inputs = tf.keras.Input(shape=input_shape)
    #Encoder
    e1 = RRU(32, t)(inputs); p1=tf.keras.layers.MaxPooling2D()(e1)
    e2 = RRU(64, t)(p1); p2=tf.keras.layers.MaxPooling2D()(e2)
    e3 = RRU(128, t)(p2); p3=tf.keras.layers.MaxPooling2D()(e3)
    e4 = RRU(256, t)(p3); p4=tf.keras.layers.MaxPooling2D()(e4)
    # Bottleneck
    b = RRU(512, t)(p4)
    # Decoder
    u1 = tf.keras.layers.UpSampling2D()(b); u1=tf.keras.layers.Concatenate()([u1,e4]); d1 = RRU(256,t)(u1)
    u2 = tf.keras.layers.UpSampling2D()(d1); u2=tf.keras.layers.Concatenate()([u2,e3]); d2 = RRU(128,t)(u2)
    u3 = tf.keras.layers.UpSampling2D()(d2); u3=tf.keras.layers.Concatenate()([u3,e2]); d3 = RRU(64,t)(u3)
    u4 = tf.keras.layers.UpSampling2D()(d3); u4=tf.keras.layers.Concatenate()([u4,e1]); d4 = RRU(32,t)(u4)
    outputs = tf.keras.layers.Conv2D(num_classes,1,activation='sigmoid')(d4)
    return tf.keras.Model(inputs, outputs)

#KFold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
all_fold_dice_scores = []

for train_idx, val_idx in kfold.split(X):
    print(f"========== Fold {fold} ==========")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    train_dataset = train_dataset.map(tf_augment, num_parallel_calls=tf.data.AUTOTUNE)
    train_dataset = train_dataset.shuffle(128).batch(16).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    val_dataset = val_dataset.batch(16).prefetch(tf.data.AUTOTUNE)

    lr_schedule = tf.keras.callbacks.LearningRateScheduler(lambda epoch: 1e-4*(1+math.cos(math.pi*epoch/150))/2)
    early_stop = tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)
    checkpoint = tf.keras.callbacks.ModelCheckpoint(f"r2unet_leakyrelu_fold{fold}.h5", save_best_only=True)

    model = build_r2unet(input_shape=(256,256,3), t=4)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=hybrid_loss, metrics=['accuracy'])
    model.fit(train_dataset, validation_data=val_dataset, epochs=150, callbacks=[lr_schedule, early_stop, checkpoint], verbose=1)

    #Fold Evaluation
    dice_scores_fold = []
    preds = model.predict(X_val, batch_size=16, verbose=0)
    preds_bin = (preds>0.5).astype(np.uint8)
    for pb, gt in zip(preds_bin, y_val):
        if np.sum(pb)>0 and np.sum(gt)>0:
            dice_scores_fold.append(binary.dc(pb.squeeze(), gt.squeeze()))
    mean_dice_fold = np.mean(dice_scores_fold) if len(dice_scores_fold)>0 else 0
    print(f"✅ Fold {fold} Dice: {mean_dice_fold:.4f}")
    all_fold_dice_scores.append(mean_dice_fold)
    fold += 1

#Average Dice
print(f"✅ Average Dice across all folds: {np.mean(all_fold_dice_scores):.4f}")

✅ Unzipped stage1_train.zip successfully!


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipython-input-1371561204.py:39: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.GridDistortion(p=0.2), A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.2)


========== Fold 1 ==========


/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Epoch 1/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7347 - loss: 0.4726   

...

34/34 ━━━━━━━━━━━━━━━━━━━━ 25s 722ms/step - accuracy: 0.9734 - loss: 0.0615 - val_accuracy: 0.9733 - val_loss: 0.0535 - learning_rate: 4.0631e-05
Epoch 86/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 698ms/step - accuracy: 0.9744 - loss: 0.0642 - val_accuracy: 0.9717 - val_loss: 0.0560 - learning_rate: 3.9604e-05
Epoch 87/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 697ms/step - accuracy: 0.9725 - loss: 0.0641 - val_accuracy: 0.9718 - val_loss: 0.0559 - learning_rate: 3.8582e-05
Epoch 88/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 696ms/step - accuracy: 0.9744 - loss: 0.0639 - val_accuracy: 0.9722 - val_loss: 0.0553 - learning_rate: 3.7566e-05
Epoch 89/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 696ms/step - accuracy: 0.9719 - loss: 0.0704 - val_accuracy: 0.9715 - val_loss: 0.0560 - learning_rate: 3.6554e-05
Epoch 90/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 698ms/step - accuracy: 0.9749 - loss: 0.0552 - val_accuracy: 0.9713 - val_loss: 0.0563 - learning_rate: 3.5548e-05
Epoch 91/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 650ms/step - acc

34/34 ━━━━━━━━━━━━━━━━━━━━ 25s 722ms/step - accuracy: 0.9746 - loss: 0.0583 - val_accuracy: 0.9735 - val_loss: 0.0534 - learning_rate: 3.4549e-05
Epoch 92/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 696ms/step - accuracy: 0.9753 - loss: 0.0587 - val_accuracy: 0.9729 - val_loss: 0.0537 - learning_rate: 3.3557e-05
Epoch 93/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 697ms/step - accuracy: 0.9717 - loss: 0.0703 - val_accuracy: 0.9730 - val_loss: 0.0542 - learning_rate: 3.2571e-05
Epoch 94/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 699ms/step - accuracy: 0.9756 - loss: 0.0587 - val_accuracy: 0.9726 - val_loss: 0.0546 - learning_rate: 3.1594e-05
Epoch 95/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 698ms/step - accuracy: 0.9751 - loss: 0.0594 - val_accuracy: 0.9726 - val_loss: 0.0544 - learning_rate: 3.0624e-05
Epoch 96/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 696ms/step - accuracy: 0.9711 - loss: 0.0741 - val_accuracy: 0.9723 - val_loss: 0.0545 - learning_rate: 2.9663e-05
Epoch 97/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 648ms/step - acc

34/34 ━━━━━━━━━━━━━━━━━━━━ 25s 719ms/step - accuracy: 0.9750 - loss: 0.0604 - val_accuracy: 0.9733 - val_loss: 0.0531 - learning_rate: 2.8711e-05
Epoch 98/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 696ms/step - accuracy: 0.9764 - loss: 0.0570 - val_accuracy: 0.9726 - val_loss: 0.0545 - learning_rate: 2.7768e-05
Epoch 99/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 699ms/step - accuracy: 0.9758 - loss: 0.0552 - val_accuracy: 0.9731 - val_loss: 0.0536 - learning_rate: 2.6835e-05
Epoch 100/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 696ms/step - accuracy: 0.9762 - loss: 0.0561 - val_accuracy: 0.9730 - val_loss: 0.0535 - learning_rate: 2.5912e-05
Epoch 101/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 648ms/step - accuracy: 0.9765 - loss: 0.0558

34/34 ━━━━━━━━━━━━━━━━━━━━ 25s 719ms/step - accuracy: 0.9764 - loss: 0.0559 - val_accuracy: 0.9734 - val_loss: 0.0530 - learning_rate: 2.5000e-05
Epoch 102/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 696ms/step - accuracy: 0.9720 - loss: 0.0661 - val_accuracy: 0.9732 - val_loss: 0.0531 - learning_rate: 2.4099e-05
Epoch 103/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 652ms/step - accuracy: 0.9749 - loss: 0.0578

34/34 ━━━━━━━━━━━━━━━━━━━━ 25s 722ms/step - accuracy: 0.9749 - loss: 0.0579 - val_accuracy: 0.9736 - val_loss: 0.0526 - learning_rate: 2.3209e-05
Epoch 104/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 696ms/step - accuracy: 0.9736 - loss: 0.0633 - val_accuracy: 0.9733 - val_loss: 0.0529 - learning_rate: 2.2330e-05
Epoch 105/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 695ms/step - accuracy: 0.9751 - loss: 0.0588 - val_accuracy: 0.9734 - val_loss: 0.0529 - learning_rate: 2.1464e-05
Epoch 106/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 698ms/step - accuracy: 0.9738 - loss: 0.0616 - val_accuracy: 0.9733 - val_loss: 0.0529 - learning_rate: 2.0611e-05
Epoch 107/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 698ms/step - accuracy: 0.9749 - loss: 0.0579 - val_accuracy: 0.9729 - val_loss: 0.0535 - learning_rate: 1.9770e-05
Epoch 108/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 694ms/step - accuracy: 0.9756 - loss: 0.0596 - val_accuracy: 0.9731 - val_loss: 0.0534 - learning_rate: 1.8943e-05
Epoch 109/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 694ms/ste

##Swish

In [ ]:
import zipfile, os, cv2, numpy as np, math
import tensorflow as tf
import albumentations as A
from sklearn.model_selection import KFold
from medpy.metric import binary

#Unzip Data
with zipfile.ZipFile('/content/stage1_train.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/stage1_train')
print("✅ Unzipped stage1_train.zip successfully!")

#Load Data
def load_dsb2018_data(dataset_dir, image_size=(256,256)):
    images, masks = [], []
    for folder in sorted(os.listdir(dataset_dir)):
        img_path = os.path.join(dataset_dir, folder, 'images', folder + '.png')
        mask_dir = os.path.join(dataset_dir, folder, 'masks')
        if not os.path.exists(img_path) or not os.path.exists(mask_dir):
            continue
        image = cv2.imread(img_path)
        image = cv2.resize(image, image_size).astype(np.float32)/255.0
        mask = np.zeros(image_size, dtype=np.uint8)
        for m in os.listdir(mask_dir):
            msk = cv2.imread(os.path.join(mask_dir, m), cv2.IMREAD_GRAYSCALE)
            msk = cv2.resize(msk, image_size)
            mask = np.maximum(mask, msk)
        mask = (mask>0).astype(np.float32)
        images.append(image)
        masks.append(np.expand_dims(mask, axis=-1))
    return np.array(images), np.array(masks)

X, y = load_dsb2018_data('/content/stage1_train', image_size=(256,256))

#Augmentation
transform = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2), A.GaussianBlur(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
    A.GridDistortion(p=0.2), A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.2)
])

def augment(image, mask):
    augmented = transform(image=image, mask=mask)
    return augmented['image'], augmented['mask']

def tf_augment(img, mask):
    img, mask = tf.numpy_function(augment, [img, mask], [tf.float32, tf.float32])
    img.set_shape([256,256,3])
    mask.set_shape([256,256,1])
    return img, mask

#Loss Function
def tversky(y_true, y_pred, alpha=0.5, beta=0.5):
    smooth = 1e-6
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    tp = tf.reduce_sum(y_true * y_pred)
    fn = tf.reduce_sum(y_true * (1 - y_pred))
    fp = tf.reduce_sum((1 - y_true) * y_pred)
    return (tp + smooth) / (tp + alpha*fn + beta*fp + smooth)

def focal_tversky_loss(y_true, y_pred, gamma=1.33):
    tv = tversky(y_true, y_pred)
    return tf.pow((1 - tv), gamma)

def dice_loss(y_true, y_pred):
    smooth=1e-6
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1 - (2.*intersection + smooth)/(tf.reduce_sum(y_true_f)+tf.reduce_sum(y_pred_f)+smooth)

def hybrid_loss(y_true, y_pred):
    return 0.7*focal_tversky_loss(y_true, y_pred) + 0.3*dice_loss(y_true, y_pred)

#R2U-Net Model
class RecurrentConv(tf.keras.layers.Layer):
    def __init__(self, filters, t=2):
        super().__init__()
        self.filters = filters
        self.t = t
        self.activation = tf.keras.layers.Activation('swish')
        self.convs = [tf.keras.layers.Conv2D(filters, 3, padding='same') for _ in range(t)]
        self.bns = [tf.keras.layers.BatchNormalization() for _ in range(t)]
    def call(self, x):
        h = 0
        for i in range(self.t):
            h = self.activation(self.bns[i](self.convs[i](x + h))) if i>0 else self.activation(self.bns[i](self.convs[i](x)))
        return h

class RRU(tf.keras.layers.Layer):
    def __init__(self, filters, t=2):
        super().__init__()
        self.projection = tf.keras.layers.Conv2D(filters, 1, padding='same')
        self.rcl = RecurrentConv(filters, t)
    def call(self, x):
        x_proj = self.projection(x)
        return x_proj + self.rcl(x_proj)

def build_r2unet(input_shape=(256,256,3), num_classes=1, t=4):
    inputs = tf.keras.Input(shape=input_shape)
    #Encoder
    e1 = RRU(32, t)(inputs); p1=tf.keras.layers.MaxPooling2D()(e1)
    e2 = RRU(64, t)(p1); p2=tf.keras.layers.MaxPooling2D()(e2)
    e3 = RRU(128, t)(p2); p3=tf.keras.layers.MaxPooling2D()(e3)
    e4 = RRU(256, t)(p3); p4=tf.keras.layers.MaxPooling2D()(e4)
    # Bottleneck
    b = RRU(512, t)(p4)
    # Decoder
    u1 = tf.keras.layers.UpSampling2D()(b); u1=tf.keras.layers.Concatenate()([u1,e4]); d1 = RRU(256,t)(u1)
    u2 = tf.keras.layers.UpSampling2D()(d1); u2=tf.keras.layers.Concatenate()([u2,e3]); d2 = RRU(128,t)(u2)
    u3 = tf.keras.layers.UpSampling2D()(d2); u3=tf.keras.layers.Concatenate()([u3,e2]); d3 = RRU(64,t)(u3)
    u4 = tf.keras.layers.UpSampling2D()(d3); u4=tf.keras.layers.Concatenate()([u4,e1]); d4 = RRU(32,t)(u4)
    outputs = tf.keras.layers.Conv2D(num_classes,1,activation='sigmoid')(d4)
    return tf.keras.Model(inputs, outputs)

#KFold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
all_fold_dice_scores = []

for train_idx, val_idx in kfold.split(X):
    print(f"========== Fold {fold} ==========")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    train_dataset = train_dataset.map(tf_augment, num_parallel_calls=tf.data.AUTOTUNE)
    train_dataset = train_dataset.shuffle(128).batch(16).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    val_dataset = val_dataset.batch(16).prefetch(tf.data.AUTOTUNE)

    lr_schedule = tf.keras.callbacks.LearningRateScheduler(lambda epoch: 1e-4*(1+math.cos(math.pi*epoch/150))/2)
    early_stop = tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)
    checkpoint = tf.keras.callbacks.ModelCheckpoint(f"r2unet_swish_fold{fold}.h5", save_best_only=True)

    model = build_r2unet(input_shape=(256,256,3), t=4)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=hybrid_loss, metrics=['accuracy'])
    model.fit(train_dataset, validation_data=val_dataset, epochs=150, callbacks=[lr_schedule, early_stop, checkpoint], verbose=1)

    #Fold Evaluation
    dice_scores_fold = []
    preds = model.predict(X_val, batch_size=16, verbose=0)
    preds_bin = (preds>0.5).astype(np.uint8)
    for pb, gt in zip(preds_bin, y_val):
        if np.sum(pb)>0 and np.sum(gt)>0:
            dice_scores_fold.append(binary.dc(pb.squeeze(), gt.squeeze()))
    mean_dice_fold = np.mean(dice_scores_fold) if len(dice_scores_fold)>0 else 0
    print(f"✅ Fold {fold} Dice: {mean_dice_fold:.4f}")
    all_fold_dice_scores.append(mean_dice_fold)
    fold += 1

#Average Dice
print(f"✅ Average Dice across all folds: {np.mean(all_fold_dice_scores):.4f}")

✅ Unzipped stage1_train.zip successfully!


/tmp/ipython-input-1108876742.py:39: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.GridDistortion(p=0.2), A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.2)


========== Fold 1 ==========
Epoch 1/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6900 - loss: 0.4763   

34/34 ━━━━━━━━━━━━━━━━━━━━ 86s 1s/step - accuracy: 0.6934 - loss: 0.4733 - val_accuracy: 0.8923 - val_loss: 0.6670 - learning_rate: 1.0000e-04
Epoch 2/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 664ms/step - accuracy: 0.9237 - loss: 0.1996 - val_accuracy: 0.8824 - val_loss: 0.7001 - learning_rate: 9.9989e-05
Epoch 3/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 672ms/step - accuracy: 0.9443 - loss: 0.1396 - val_accuracy: 0.8767 - val_loss: 0.8579 - learning_rate: 9.9956e-05
Epoch 4/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 660ms/step - accuracy: 0.9499 - loss: 0.1165 - val_accuracy: 0.8766 - val_loss: 0.8743 - learning_rate: 9.9901e-05
Epoch 5/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 22s 656ms/step - accuracy: 0.9512 - loss: 0.1184 - val_accuracy: 0.8768 - val_loss: 0.8770 - learning_rate: 9.9825e-05
Epoch 6/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 660ms/step - accuracy: 0.9536 - loss: 0.1117 - val_accuracy: 0.8754 - val_loss: 0.9154 - learning_rate: 9.9726e-05
Epoch 7/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 664ms/step - accuracy: 0

34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 687ms/step - accuracy: 0.9526 - loss: 0.1189 - val_accuracy: 0.8923 - val_loss: 0.6358 - learning_rate: 9.9464e-05
Epoch 9/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 612ms/step - accuracy: 0.9598 - loss: 0.0972

...

34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 687ms/step - accuracy: 0.9759 - loss: 0.0552 - val_accuracy: 0.9734 - val_loss: 0.0530 - learning_rate: 2.4099e-05
Epoch 103/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 664ms/step - accuracy: 0.9760 - loss: 0.0609 - val_accuracy: 0.9730 - val_loss: 0.0537 - learning_rate: 2.3209e-05
Epoch 104/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 662ms/step - accuracy: 0.9757 - loss: 0.0541 - val_accuracy: 0.9730 - val_loss: 0.0534 - learning_rate: 2.2330e-05
Epoch 105/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 661ms/step - accuracy: 0.9757 - loss: 0.0549 - val_accuracy: 0.9733 - val_loss: 0.0533 - learning_rate: 2.1464e-05
Epoch 106/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 660ms/step - accuracy: 0.9751 - loss: 0.0587 - val_accuracy: 0.9730 - val_loss: 0.0537 - learning_rate: 2.0611e-05
Epoch 107/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 662ms/step - accuracy: 0.9732 - loss: 0.0651 - val_accuracy: 0.9735 - val_loss: 0.0532 - learning_rate: 1.9770e-05
Epoch 108/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 662ms/ste

34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 686ms/step - accuracy: 0.9743 - loss: 0.0611 - val_accuracy: 0.9735 - val_loss: 0.0529 - learning_rate: 1.4276e-05
Epoch 115/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 662ms/step - accuracy: 0.9760 - loss: 0.0606 - val_accuracy: 0.9728 - val_loss: 0.0539 - learning_rate: 1.3552e-05
Epoch 116/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 615ms/step - accuracy: 0.9763 - loss: 0.0558

34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 687ms/step - accuracy: 0.9762 - loss: 0.0559 - val_accuracy: 0.9734 - val_loss: 0.0529 - learning_rate: 1.2843e-05
Epoch 117/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 663ms/step - accuracy: 0.9758 - loss: 0.0573 - val_accuracy: 0.9732 - val_loss: 0.0536 - learning_rate: 1.2150e-05
Epoch 118/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 663ms/step - accuracy: 0.9742 - loss: 0.0634 - val_accuracy: 0.9730 - val_loss: 0.0534 - learning_rate: 1.1474e-05
Epoch 119/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 663ms/step - accuracy: 0.9743 - loss: 0.0593 - val_accuracy: 0.9730 - val_loss: 0.0533 - learning_rate: 1.0815e-05
Epoch 120/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 664ms/step - accuracy: 0.9754 - loss: 0.0595 - val_accuracy: 0.9730 - val_loss: 0.0539 - learning_rate: 1.0174e-05
Epoch 121/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 616ms/step - accuracy: 0.9745 - loss: 0.0603

34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 687ms/step - accuracy: 0.9745 - loss: 0.0603 - val_accuracy: 0.9735 - val_loss: 0.0525 - learning_rate: 9.5492e-06
Epoch 122/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 616ms/step - accuracy: 0.9767 - loss: 0.0553

34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 688ms/step - accuracy: 0.9766 - loss: 0.0553 - val_accuracy: 0.9737 - val_loss: 0.0523 - learning_rate: 8.9425e-06
Epoch 123/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 662ms/step - accuracy: 0.9762 - loss: 0.0548 - val_accuracy: 0.9736 - val_loss: 0.0525 - learning_rate: 8.3539e-06
Epoch 124/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 662ms/step - accuracy: 0.9752 - loss: 0.0616 - val_accuracy: 0.9734 - val_loss: 0.0529 - learning_rate: 7.7836e-06
Epoch 125/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 663ms/step - accuracy: 0.9769 - loss: 0.0554 - val_accuracy: 0.9734 - val_loss: 0.0531 - learning_rate: 7.2318e-06
Epoch 126/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 663ms/step - accuracy: 0.9755 - loss: 0.0564 - val_accuracy: 0.9734 - val_loss: 0.0528 - learning_rate: 6.6987e-06
Epoch 127/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 662ms/step - accuracy: 0.9766 - loss: 0.0527 - val_accuracy: 0.9733 - val_loss: 0.0530 - learning_rate: 6.1847e-06
Epoch 128/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 41s 660ms/ste

In [ ]:
from google.colab import files

for fold in range(1, 6):
    files.download(f"r2unet_swish_fold{fold}.h5")

##GeLU

In [ ]:
import zipfile, os, cv2, numpy as np, math
import tensorflow as tf
import albumentations as A
from sklearn.model_selection import KFold
from medpy.metric import binary

#Unzip Data
with zipfile.ZipFile('/content/stage1_train.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/stage1_train')
print("✅ Unzipped stage1_train.zip successfully!")

#Load Data
def load_dsb2018_data(dataset_dir, image_size=(256,256)):
    images, masks = [], []
    for folder in sorted(os.listdir(dataset_dir)):
        img_path = os.path.join(dataset_dir, folder, 'images', folder + '.png')
        mask_dir = os.path.join(dataset_dir, folder, 'masks')
        if not os.path.exists(img_path) or not os.path.exists(mask_dir):
            continue
        image = cv2.imread(img_path)
        image = cv2.resize(image, image_size).astype(np.float32)/255.0
        mask = np.zeros(image_size, dtype=np.uint8)
        for m in os.listdir(mask_dir):
            msk = cv2.imread(os.path.join(mask_dir, m), cv2.IMREAD_GRAYSCALE)
            msk = cv2.resize(msk, image_size)
            mask = np.maximum(mask, msk)
        mask = (mask>0).astype(np.float32)
        images.append(image)
        masks.append(np.expand_dims(mask, axis=-1))
    return np.array(images), np.array(masks)

X, y = load_dsb2018_data('/content/stage1_train', image_size=(256,256))

#Augmentation
transform = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2), A.GaussianBlur(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
    A.GridDistortion(p=0.2), A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.2)
])

def augment(image, mask):
    augmented = transform(image=image, mask=mask)
    return augmented['image'], augmented['mask']

def tf_augment(img, mask):
    img, mask = tf.numpy_function(augment, [img, mask], [tf.float32, tf.float32])
    img.set_shape([256,256,3])
    mask.set_shape([256,256,1])
    return img, mask

#Loss Function
def tversky(y_true, y_pred, alpha=0.5, beta=0.5):
    smooth = 1e-6
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    tp = tf.reduce_sum(y_true * y_pred)
    fn = tf.reduce_sum(y_true * (1 - y_pred))
    fp = tf.reduce_sum((1 - y_true) * y_pred)
    return (tp + smooth) / (tp + alpha*fn + beta*fp + smooth)

def focal_tversky_loss(y_true, y_pred, gamma=1.33):
    tv = tversky(y_true, y_pred)
    return tf.pow((1 - tv), gamma)

def dice_loss(y_true, y_pred):
    smooth=1e-6
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1 - (2.*intersection + smooth)/(tf.reduce_sum(y_true_f)+tf.reduce_sum(y_pred_f)+smooth)

def hybrid_loss(y_true, y_pred):
    return 0.7*focal_tversky_loss(y_true, y_pred) + 0.3*dice_loss(y_true, y_pred)

#R2U-Net Model
class RecurrentConv(tf.keras.layers.Layer):
    def __init__(self, filters, t=2):
        super().__init__()
        self.filters = filters
        self.t = t
        self.activation = tf.keras.layers.Activation('gelu')
        self.convs = [tf.keras.layers.Conv2D(filters, 3, padding='same') for _ in range(t)]
        self.bns = [tf.keras.layers.BatchNormalization() for _ in range(t)]
    def call(self, x):
        h = 0
        for i in range(self.t):
            h = self.activation(self.bns[i](self.convs[i](x + h))) if i>0 else self.activation(self.bns[i](self.convs[i](x)))
        return h

class RRU(tf.keras.layers.Layer):
    def __init__(self, filters, t=2):
        super().__init__()
        self.projection = tf.keras.layers.Conv2D(filters, 1, padding='same')
        self.rcl = RecurrentConv(filters, t)
    def call(self, x):
        x_proj = self.projection(x)
        return x_proj + self.rcl(x_proj)

def build_r2unet(input_shape=(256,256,3), num_classes=1, t=4):
    inputs = tf.keras.Input(shape=input_shape)
    #Encoder
    e1 = RRU(32, t)(inputs); p1=tf.keras.layers.MaxPooling2D()(e1)
    e2 = RRU(64, t)(p1); p2=tf.keras.layers.MaxPooling2D()(e2)
    e3 = RRU(128, t)(p2); p3=tf.keras.layers.MaxPooling2D()(e3)
    e4 = RRU(256, t)(p3); p4=tf.keras.layers.MaxPooling2D()(e4)
    # Bottleneck
    b = RRU(512, t)(p4)
    # Decoder
    u1 = tf.keras.layers.UpSampling2D()(b); u1=tf.keras.layers.Concatenate()([u1,e4]); d1 = RRU(256,t)(u1)
    u2 = tf.keras.layers.UpSampling2D()(d1); u2=tf.keras.layers.Concatenate()([u2,e3]); d2 = RRU(128,t)(u2)
    u3 = tf.keras.layers.UpSampling2D()(d2); u3=tf.keras.layers.Concatenate()([u3,e2]); d3 = RRU(64,t)(u3)
    u4 = tf.keras.layers.UpSampling2D()(d3); u4=tf.keras.layers.Concatenate()([u4,e1]); d4 = RRU(32,t)(u4)
    outputs = tf.keras.layers.Conv2D(num_classes,1,activation='sigmoid')(d4)
    return tf.keras.Model(inputs, outputs)

#KFold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
all_fold_dice_scores = []

for train_idx, val_idx in kfold.split(X):
    print(f"========== Fold {fold} ==========")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    train_dataset = train_dataset.map(tf_augment, num_parallel_calls=tf.data.AUTOTUNE)
    train_dataset = train_dataset.shuffle(128).batch(16).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    val_dataset = val_dataset.batch(16).prefetch(tf.data.AUTOTUNE)

    lr_schedule = tf.keras.callbacks.LearningRateScheduler(lambda epoch: 1e-4*(1+math.cos(math.pi*epoch/150))/2)
    early_stop = tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)
    checkpoint = tf.keras.callbacks.ModelCheckpoint(f"r2unet_gelu_fold{fold}.h5", save_best_only=True)

    model = build_r2unet(input_shape=(256,256,3), t=4)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=hybrid_loss, metrics=['accuracy'])
    model.fit(train_dataset, validation_data=val_dataset, epochs=150, callbacks=[lr_schedule, early_stop, checkpoint], verbose=1)

    #Fold Evaluation
    dice_scores_fold = []
    preds = model.predict(X_val, batch_size=16, verbose=0)
    preds_bin = (preds>0.5).astype(np.uint8)
    for pb, gt in zip(preds_bin, y_val):
        if np.sum(pb)>0 and np.sum(gt)>0:
            dice_scores_fold.append(binary.dc(pb.squeeze(), gt.squeeze()))
    mean_dice_fold = np.mean(dice_scores_fold) if len(dice_scores_fold)>0 else 0
    print(f"✅ Fold {fold} Dice: {mean_dice_fold:.4f}")
    all_fold_dice_scores.append(mean_dice_fold)
    fold += 1

#Average Dice
print(f"✅ Average Dice across all folds: {np.mean(all_fold_dice_scores):.4f}")

✅ Unzipped stage1_train.zip successfully!


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipython-input-3652039019.py:39: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.GridDistortion(p=0.2), A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.2)


========== Fold 1 ==========
Epoch 1/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8035 - loss: 0.3922   

34/34 ━━━━━━━━━━━━━━━━━━━━ 117s 2s/step - accuracy: 0.8055 - loss: 0.3891 - val_accuracy: 0.2581 - val_loss: 0.7128 - learning_rate: 1.0000e-04
Epoch 2/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 21s 621ms/step - accuracy: 0.9232 - loss: 0.1893 - val_accuracy: 0.8724 - val_loss: 0.8796 - learning_rate: 9.9989e-05
Epoch 3/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 22s 630ms/step - accuracy: 0.9422 - loss: 0.1479 - val_accuracy: 0.8720 - val_loss: 0.9936 - learning_rate: 9.9956e-05
Epoch 4/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 22s 637ms/step - accuracy: 0.9490 - loss: 0.1268 - val_accuracy: 0.8720 - val_loss: 0.9999 - learning_rate: 9.9901e-05
Epoch 5/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 22s 642ms/step - accuracy: 0.9515 - loss: 0.1183 - val_accuracy: 0.8721 - val_loss: 0.9968 - learning_rate: 9.9825e-05
Epoch 6/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 22s 649ms/step - accuracy: 0.9504 - loss: 0.1199 - val_accuracy: 0.8720 - val_loss: 0.9997 - learning_rate: 9.9726e-05
Epoch 7/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 22s 656ms/step - accuracy: 

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 692ms/step - accuracy: 0.9606 - loss: 0.0956 - val_accuracy: 0.9037 - val_loss: 0.4973 - learning_rate: 9.9114e-05
Epoch 11/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 625ms/step - accuracy: 0.9589 - loss: 0.0950

...

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 709ms/step - accuracy: 0.9746 - loss: 0.0627 - val_accuracy: 0.9721 - val_loss: 0.0557 - learning_rate: 6.4452e-05
Epoch 63/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 685ms/step - accuracy: 0.9723 - loss: 0.0667 - val_accuracy: 0.9704 - val_loss: 0.0584 - learning_rate: 6.3446e-05
Epoch 64/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 639ms/step - accuracy: 0.9715 - loss: 0.0676

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 709ms/step - accuracy: 0.9716 - loss: 0.0675 - val_accuracy: 0.9722 - val_loss: 0.0555 - learning_rate: 6.2434e-05
Epoch 65/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 685ms/step - accuracy: 0.9725 - loss: 0.0671 - val_accuracy: 0.9720 - val_loss: 0.0562 - learning_rate: 6.1418e-05
Epoch 66/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 685ms/step - accuracy: 0.9741 - loss: 0.0659 - val_accuracy: 0.9714 - val_loss: 0.0567 - learning_rate: 6.0396e-05
Epoch 67/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 639ms/step - accuracy: 0.9727 - loss: 0.0728

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 710ms/step - accuracy: 0.9727 - loss: 0.0726 - val_accuracy: 0.9729 - val_loss: 0.0546 - learning_rate: 5.9369e-05
Epoch 68/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 685ms/step - accuracy: 0.9724 - loss: 0.0639 - val_accuracy: 0.9720 - val_loss: 0.0554 - learning_rate: 5.8338e-05
Epoch 69/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 686ms/step - accuracy: 0.9707 - loss: 0.0693 - val_accuracy: 0.9728 - val_loss: 0.0550 - learning_rate: 5.7304e-05
Epoch 70/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 686ms/step - accuracy: 0.9717 - loss: 0.0673 - val_accuracy: 0.9719 - val_loss: 0.0558 - learning_rate: 5.6267e-05
Epoch 71/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 686ms/step - accuracy: 0.9728 - loss: 0.0651 - val_accuracy: 0.9712 - val_loss: 0.0571 - learning_rate: 5.5226e-05
Epoch 72/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 686ms/step - accuracy: 0.9719 - loss: 0.0662 - val_accuracy: 0.9724 - val_loss: 0.0563 - learning_rate: 5.4184e-05
Epoch 73/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 685ms/step - ac

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 708ms/step - accuracy: 0.9735 - loss: 0.0634 - val_accuracy: 0.9732 - val_loss: 0.0540 - learning_rate: 4.7906e-05
Epoch 79/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 686ms/step - accuracy: 0.9734 - loss: 0.0612 - val_accuracy: 0.9712 - val_loss: 0.0567 - learning_rate: 4.6860e-05
Epoch 80/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 685ms/step - accuracy: 0.9731 - loss: 0.0649 - val_accuracy: 0.9729 - val_loss: 0.0547 - learning_rate: 4.5816e-05
Epoch 81/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 686ms/step - accuracy: 0.9752 - loss: 0.0604 - val_accuracy: 0.9733 - val_loss: 0.0541 - learning_rate: 4.4774e-05
Epoch 82/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 685ms/step - accuracy: 0.9741 - loss: 0.0594 - val_accuracy: 0.9669 - val_loss: 0.0640 - learning_rate: 4.3733e-05
Epoch 83/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 686ms/step - accuracy: 0.9744 - loss: 0.0578 - val_accuracy: 0.9712 - val_loss: 0.0569 - learning_rate: 4.2696e-05
Epoch 84/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 686ms/step - ac

34/34 ━━━━━━━━━━━━━━━━━━━━ 41s 696ms/step - accuracy: 0.9734 - loss: 0.0671 - val_accuracy: 0.9734 - val_loss: 0.0528 - learning_rate: 3.9604e-05
Epoch 87/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 680ms/step - accuracy: 0.9736 - loss: 0.0626 - val_accuracy: 0.9716 - val_loss: 0.0565 - learning_rate: 3.8582e-05
Epoch 88/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 681ms/step - accuracy: 0.9761 - loss: 0.0560 - val_accuracy: 0.9711 - val_loss: 0.0581 - learning_rate: 3.7566e-05
Epoch 89/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 683ms/step - accuracy: 0.9718 - loss: 0.0680 - val_accuracy: 0.9721 - val_loss: 0.0547 - learning_rate: 3.6554e-05
Epoch 90/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 684ms/step - accuracy: 0.9734 - loss: 0.0628 - val_accuracy: 0.9727 - val_loss: 0.0537 - learning_rate: 3.5548e-05
Epoch 91/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 685ms/step - accuracy: 0.9736 - loss: 0.0631 - val_accuracy: 0.9731 - val_loss: 0.0538 - learning_rate: 3.4549e-05
Epoch 92/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 684ms/step - ac

In [ ]:
from google.colab import files

for fold in range(1, 6):
    files.download(f"r2unet_gelu_fold{fold}.h5")

##SeLU

In [ ]:
import zipfile, os, cv2, numpy as np, math
import tensorflow as tf
import albumentations as A
from sklearn.model_selection import KFold
from medpy.metric import binary

#Unzip Data
with zipfile.ZipFile('/content/stage1_train.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/stage1_train')
print("✅ Unzipped stage1_train.zip successfully!")

#Load Data
def load_dsb2018_data(dataset_dir, image_size=(256,256)):
    images, masks = [], []
    for folder in sorted(os.listdir(dataset_dir)):
        img_path = os.path.join(dataset_dir, folder, 'images', folder + '.png')
        mask_dir = os.path.join(dataset_dir, folder, 'masks')
        if not os.path.exists(img_path) or not os.path.exists(mask_dir):
            continue
        image = cv2.imread(img_path)
        image = cv2.resize(image, image_size).astype(np.float32)/255.0
        mask = np.zeros(image_size, dtype=np.uint8)
        for m in os.listdir(mask_dir):
            msk = cv2.imread(os.path.join(mask_dir, m), cv2.IMREAD_GRAYSCALE)
            msk = cv2.resize(msk, image_size)
            mask = np.maximum(mask, msk)
        mask = (mask>0).astype(np.float32)
        images.append(image)
        masks.append(np.expand_dims(mask, axis=-1))
    return np.array(images), np.array(masks)

X, y = load_dsb2018_data('/content/stage1_train', image_size=(256,256))

#Augmentation
transform = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2), A.GaussianBlur(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
    A.GridDistortion(p=0.2), A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.2)
])

def augment(image, mask):
    augmented = transform(image=image, mask=mask)
    return augmented['image'], augmented['mask']

def tf_augment(img, mask):
    img, mask = tf.numpy_function(augment, [img, mask], [tf.float32, tf.float32])
    img.set_shape([256,256,3])
    mask.set_shape([256,256,1])
    return img, mask

#Loss Function
def tversky(y_true, y_pred, alpha=0.5, beta=0.5):
    smooth = 1e-6
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    tp = tf.reduce_sum(y_true * y_pred)
    fn = tf.reduce_sum(y_true * (1 - y_pred))
    fp = tf.reduce_sum((1 - y_true) * y_pred)
    return (tp + smooth) / (tp + alpha*fn + beta*fp + smooth)

def focal_tversky_loss(y_true, y_pred, gamma=1.33):
    tv = tversky(y_true, y_pred)
    return tf.pow((1 - tv), gamma)

def dice_loss(y_true, y_pred):
    smooth=1e-6
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1 - (2.*intersection + smooth)/(tf.reduce_sum(y_true_f)+tf.reduce_sum(y_pred_f)+smooth)

def hybrid_loss(y_true, y_pred):
    return 0.7*focal_tversky_loss(y_true, y_pred) + 0.3*dice_loss(y_true, y_pred)

#R2U-Net Model
class RecurrentConv(tf.keras.layers.Layer):
    def __init__(self, filters, t=2):
        super().__init__()
        self.filters = filters
        self.t = t
        self.activation = tf.keras.layers.Activation('selu')
        self.convs = [tf.keras.layers.Conv2D(filters, 3, padding='same') for _ in range(t)]
        self.bns = [tf.keras.layers.BatchNormalization() for _ in range(t)]
    def call(self, x):
        h = 0
        for i in range(self.t):
            h = self.activation(self.bns[i](self.convs[i](x + h))) if i>0 else self.activation(self.bns[i](self.convs[i](x)))
        return h

class RRU(tf.keras.layers.Layer):
    def __init__(self, filters, t=2):
        super().__init__()
        self.projection = tf.keras.layers.Conv2D(filters, 1, padding='same')
        self.rcl = RecurrentConv(filters, t)
    def call(self, x):
        x_proj = self.projection(x)
        return x_proj + self.rcl(x_proj)

def build_r2unet(input_shape=(256,256,3), num_classes=1, t=4):
    inputs = tf.keras.Input(shape=input_shape)
    #Encoder
    e1 = RRU(32, t)(inputs); p1=tf.keras.layers.MaxPooling2D()(e1)
    e2 = RRU(64, t)(p1); p2=tf.keras.layers.MaxPooling2D()(e2)
    e3 = RRU(128, t)(p2); p3=tf.keras.layers.MaxPooling2D()(e3)
    e4 = RRU(256, t)(p3); p4=tf.keras.layers.MaxPooling2D()(e4)
    # Bottleneck
    b = RRU(512, t)(p4)
    # Decoder
    u1 = tf.keras.layers.UpSampling2D()(b); u1=tf.keras.layers.Concatenate()([u1,e4]); d1 = RRU(256,t)(u1)
    u2 = tf.keras.layers.UpSampling2D()(d1); u2=tf.keras.layers.Concatenate()([u2,e3]); d2 = RRU(128,t)(u2)
    u3 = tf.keras.layers.UpSampling2D()(d2); u3=tf.keras.layers.Concatenate()([u3,e2]); d3 = RRU(64,t)(u3)
    u4 = tf.keras.layers.UpSampling2D()(d3); u4=tf.keras.layers.Concatenate()([u4,e1]); d4 = RRU(32,t)(u4)
    outputs = tf.keras.layers.Conv2D(num_classes,1,activation='sigmoid')(d4)
    return tf.keras.Model(inputs, outputs)

#KFold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
all_fold_dice_scores = []

for train_idx, val_idx in kfold.split(X):
    print(f"========== Fold {fold} ==========")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    train_dataset = train_dataset.map(tf_augment, num_parallel_calls=tf.data.AUTOTUNE)
    train_dataset = train_dataset.shuffle(128).batch(16).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    val_dataset = val_dataset.batch(16).prefetch(tf.data.AUTOTUNE)

    lr_schedule = tf.keras.callbacks.LearningRateScheduler(lambda epoch: 1e-4*(1+math.cos(math.pi*epoch/150))/2)
    early_stop = tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)
    checkpoint = tf.keras.callbacks.ModelCheckpoint(f"r2unet_selu_fold{fold}.h5", save_best_only=True)

    model = build_r2unet(input_shape=(256,256,3), t=4)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=hybrid_loss, metrics=['accuracy'])
    model.fit(train_dataset, validation_data=val_dataset, epochs=150, callbacks=[lr_schedule, early_stop, checkpoint], verbose=1)

    #Fold Evaluation
    dice_scores_fold = []
    preds = model.predict(X_val, batch_size=16, verbose=0)
    preds_bin = (preds>0.5).astype(np.uint8)
    for pb, gt in zip(preds_bin, y_val):
        if np.sum(pb)>0 and np.sum(gt)>0:
            dice_scores_fold.append(binary.dc(pb.squeeze(), gt.squeeze()))
    mean_dice_fold = np.mean(dice_scores_fold) if len(dice_scores_fold)>0 else 0
    print(f"✅ Fold {fold} Dice: {mean_dice_fold:.4f}")
    all_fold_dice_scores.append(mean_dice_fold)
    fold += 1

#Average Dice
print(f"✅ Average Dice across all folds: {np.mean(all_fold_dice_scores):.4f}")

✅ Unzipped stage1_train.zip successfully!


/tmp/ipython-input-1906753312.py:39: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.GridDistortion(p=0.2), A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.2)


========== Fold 1 ==========
Epoch 1/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7010 - loss: 0.4485   

34/34 ━━━━━━━━━━━━━━━━━━━━ 78s 1s/step - accuracy: 0.7042 - loss: 0.4452 - val_accuracy: 0.2794 - val_loss: 0.7376 - learning_rate: 1.0000e-04
Epoch 2/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 596ms/step - accuracy: 0.9280 - loss: 0.1779

34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 662ms/step - accuracy: 0.9283 - loss: 0.1774 - val_accuracy: 0.7299 - val_loss: 0.5550 - learning_rate: 9.9989e-05
Epoch 3/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 602ms/step - accuracy: 0.9499 - loss: 0.1198

...

34/34 ━━━━━━━━━━━━━━━━━━━━ 25s 718ms/step - accuracy: 0.9738 - loss: 0.0681 - val_accuracy: 0.9730 - val_loss: 0.0541 - learning_rate: 5.6898e-06
Epoch 129/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 692ms/step - accuracy: 0.9715 - loss: 0.0672 - val_accuracy: 0.9728 - val_loss: 0.0545 - learning_rate: 5.2144e-06
Epoch 130/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 693ms/step - accuracy: 0.9717 - loss: 0.0680 - val_accuracy: 0.9730 - val_loss: 0.0541 - learning_rate: 4.7586e-06
Epoch 131/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 693ms/step - accuracy: 0.9747 - loss: 0.0628 - val_accuracy: 0.9730 - val_loss: 0.0542 - learning_rate: 4.3227e-06
Epoch 132/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 694ms/step - accuracy: 0.9723 - loss: 0.0662 - val_accuracy: 0.9731 - val_loss: 0.0542 - learning_rate: 3.9068e-06
Epoch 133/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 692ms/step - accuracy: 0.9721 - loss: 0.0688 - val_accuracy: 0.9729 - val_loss: 0.0541 - learning_rate: 3.5112e-06
Epoch 134/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 647ms/step

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 718ms/step - accuracy: 0.9745 - loss: 0.0625 - val_accuracy: 0.9731 - val_loss: 0.0539 - learning_rate: 3.1359e-06
Epoch 135/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 691ms/step - accuracy: 0.9749 - loss: 0.0604 - val_accuracy: 0.9731 - val_loss: 0.0539 - learning_rate: 2.7812e-06
Epoch 136/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 646ms/step - accuracy: 0.9697 - loss: 0.0705

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 716ms/step - accuracy: 0.9697 - loss: 0.0705 - val_accuracy: 0.9731 - val_loss: 0.0539 - learning_rate: 2.4472e-06
Epoch 137/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 646ms/step - accuracy: 0.9739 - loss: 0.0611

34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 716ms/step - accuracy: 0.9739 - loss: 0.0612 - val_accuracy: 0.9731 - val_loss: 0.0539 - learning_rate: 2.1340e-06
Epoch 138/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 692ms/step - accuracy: 0.9728 - loss: 0.0671 - val_accuracy: 0.9730 - val_loss: 0.0539 - learning_rate: 1.8419e-06
Epoch 139/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 648ms/step - accuracy: 0.9754 - loss: 0.0561

34/34 ━━━━━━━━━━━━━━━━━━━━ 25s 718ms/step - accuracy: 0.9754 - loss: 0.0561 - val_accuracy: 0.9731 - val_loss: 0.0537 - learning_rate: 1.5708e-06
Epoch 140/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 693ms/step - accuracy: 0.9742 - loss: 0.0597 - val_accuracy: 0.9730 - val_loss: 0.0539 - learning_rate: 1.3211e-06
Epoch 141/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 694ms/step - accuracy: 0.9739 - loss: 0.0711 - val_accuracy: 0.9730 - val_loss: 0.0539 - learning_rate: 1.0926e-06
Epoch 142/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 694ms/step - accuracy: 0.9750 - loss: 0.0592 - val_accuracy: 0.9731 - val_loss: 0.0538 - learning_rate: 8.8564e-07
Epoch 143/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 693ms/step - accuracy: 0.9733 - loss: 0.0649 - val_accuracy: 0.9731 - val_loss: 0.0538 - learning_rate: 7.0020e-07
Epoch 144/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 692ms/step - accuracy: 0.9732 - loss: 0.0669 - val_accuracy: 0.9731 - val_loss: 0.0539 - learning_rate: 5.3638e-07
Epoch 145/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 692ms/ste

In [ ]:
from google.colab import files

for fold in range(1, 6):
    files.download(f"r2unet_selu_fold{fold}.h5")

#Recurrence Steps (t) = 3

##ReLU

###Loss: Focal Tversky + Dice

In [ ]:
!pip install medpy --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 10.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 48.0 MB/s eta 0:00:00


In [ ]:
import zipfile, os, cv2, numpy as np, math
import tensorflow as tf
import albumentations as A
from sklearn.model_selection import KFold
from medpy.metric import binary

#Unzip Data
with zipfile.ZipFile('/content/stage1_train.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/stage1_train')
print("✅ Unzipped stage1_train.zip successfully!")

#Load Data
def load_dsb2018_data(dataset_dir, image_size=(256,256)):
    images, masks = [], []
    for folder in sorted(os.listdir(dataset_dir)):
        img_path = os.path.join(dataset_dir, folder, 'images', folder + '.png')
        mask_dir = os.path.join(dataset_dir, folder, 'masks')
        if not os.path.exists(img_path) or not os.path.exists(mask_dir):
            continue
        image = cv2.imread(img_path)
        image = cv2.resize(image, image_size).astype(np.float32)/255.0
        mask = np.zeros(image_size, dtype=np.uint8)
        for m in os.listdir(mask_dir):
            msk = cv2.imread(os.path.join(mask_dir, m), cv2.IMREAD_GRAYSCALE)
            msk = cv2.resize(msk, image_size)
            mask = np.maximum(mask, msk)
        mask = (mask>0).astype(np.float32)
        images.append(image)
        masks.append(np.expand_dims(mask, axis=-1))
    return np.array(images), np.array(masks)

X, y = load_dsb2018_data('/content/stage1_train', image_size=(256,256))

#Augmentation
transform = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2), A.GaussianBlur(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
    A.GridDistortion(p=0.2), A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.2)
])

def augment(image, mask):
    augmented = transform(image=image, mask=mask)
    return augmented['image'], augmented['mask']

def tf_augment(img, mask):
    img, mask = tf.numpy_function(augment, [img, mask], [tf.float32, tf.float32])
    img.set_shape([256,256,3])
    mask.set_shape([256,256,1])
    return img, mask

#Loss Function
def tversky(y_true, y_pred, alpha=0.5, beta=0.5):
    smooth = 1e-6
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    tp = tf.reduce_sum(y_true * y_pred)
    fn = tf.reduce_sum(y_true * (1 - y_pred))
    fp = tf.reduce_sum((1 - y_true) * y_pred)
    return (tp + smooth) / (tp + alpha*fn + beta*fp + smooth)

def focal_tversky_loss(y_true, y_pred, gamma=1.33):
    tv = tversky(y_true, y_pred)
    return tf.pow((1 - tv), gamma)

def dice_loss(y_true, y_pred):
    smooth=1e-6
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1 - (2.*intersection + smooth)/(tf.reduce_sum(y_true_f)+tf.reduce_sum(y_pred_f)+smooth)

def hybrid_loss(y_true, y_pred):
    return 0.7*focal_tversky_loss(y_true, y_pred) + 0.3*dice_loss(y_true, y_pred)

#R2U-Net Model
class RecurrentConv(tf.keras.layers.Layer):
    def __init__(self, filters, t=2):
        super().__init__()
        self.filters = filters
        self.t = t
        self.activation = tf.keras.layers.Activation('relu')
        self.convs = [tf.keras.layers.Conv2D(filters, 3, padding='same') for _ in range(t)]
        self.bns = [tf.keras.layers.BatchNormalization() for _ in range(t)]
    def call(self, x):
        h = 0
        for i in range(self.t):
            h = self.activation(self.bns[i](self.convs[i](x + h))) if i>0 else self.activation(self.bns[i](self.convs[i](x)))
        return h

class RRU(tf.keras.layers.Layer):
    def __init__(self, filters, t=2):
        super().__init__()
        self.projection = tf.keras.layers.Conv2D(filters, 1, padding='same')
        self.rcl = RecurrentConv(filters, t)
    def call(self, x):
        x_proj = self.projection(x)
        return x_proj + self.rcl(x_proj)

def build_r2unet(input_shape=(256,256,3), num_classes=1, t=4):
    inputs = tf.keras.Input(shape=input_shape)
    #Encoder
    e1 = RRU(32, t)(inputs); p1=tf.keras.layers.MaxPooling2D()(e1)
    e2 = RRU(64, t)(p1); p2=tf.keras.layers.MaxPooling2D()(e2)
    e3 = RRU(128, t)(p2); p3=tf.keras.layers.MaxPooling2D()(e3)
    e4 = RRU(256, t)(p3); p4=tf.keras.layers.MaxPooling2D()(e4)
    # Bottleneck
    b = RRU(512, t)(p4)
    # Decoder
    u1 = tf.keras.layers.UpSampling2D()(b); u1=tf.keras.layers.Concatenate()([u1,e4]); d1 = RRU(256,t)(u1)
    u2 = tf.keras.layers.UpSampling2D()(d1); u2=tf.keras.layers.Concatenate()([u2,e3]); d2 = RRU(128,t)(u2)
    u3 = tf.keras.layers.UpSampling2D()(d2); u3=tf.keras.layers.Concatenate()([u3,e2]); d3 = RRU(64,t)(u3)
    u4 = tf.keras.layers.UpSampling2D()(d3); u4=tf.keras.layers.Concatenate()([u4,e1]); d4 = RRU(32,t)(u4)
    outputs = tf.keras.layers.Conv2D(num_classes,1,activation='sigmoid')(d4)
    return tf.keras.Model(inputs, outputs)

#KFold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
all_fold_dice_scores = []

for train_idx, val_idx in kfold.split(X):
    print(f"========== Fold {fold} ==========")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    train_dataset = train_dataset.map(tf_augment, num_parallel_calls=tf.data.AUTOTUNE)
    train_dataset = train_dataset.shuffle(128).batch(16).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    val_dataset = val_dataset.batch(16).prefetch(tf.data.AUTOTUNE)

    lr_schedule = tf.keras.callbacks.LearningRateScheduler(lambda epoch: 1e-4*(1+math.cos(math.pi*epoch/150))/2)
    early_stop = tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)
    checkpoint = tf.keras.callbacks.ModelCheckpoint(f"r2unet_relu_t3_fold{fold}.h5", save_best_only=True)

    model = build_r2unet(input_shape=(256,256,3), t=3)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=hybrid_loss, metrics=['accuracy'])
    model.fit(train_dataset, validation_data=val_dataset, epochs=150, callbacks=[lr_schedule, early_stop, checkpoint], verbose=1)

    #Fold Evaluation
    dice_scores_fold = []
    preds = model.predict(X_val, batch_size=16, verbose=0)
    preds_bin = (preds>0.5).astype(np.uint8)
    for pb, gt in zip(preds_bin, y_val):
        if np.sum(pb)>0 and np.sum(gt)>0:
            dice_scores_fold.append(binary.dc(pb.squeeze(), gt.squeeze()))
    mean_dice_fold = np.mean(dice_scores_fold) if len(dice_scores_fold)>0 else 0
    print(f"✅ Fold {fold} Dice: {mean_dice_fold:.4f}")
    all_fold_dice_scores.append(mean_dice_fold)
    fold += 1

#Average Dice
print(f"✅ Average Dice across all folds: {np.mean(all_fold_dice_scores):.4f}")

✅ Unzipped stage1_train.zip successfully!


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipython-input-295928933.py:39: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.GridDistortion(p=0.2), A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.2)


========== Fold 1 ==========
Epoch 1/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6451 - loss: 0.5262   

34/34 ━━━━━━━━━━━━━━━━━━━━ 90s 1s/step - accuracy: 0.6484 - loss: 0.5231 - val_accuracy: 0.8831 - val_loss: 0.6832 - learning_rate: 1.0000e-04
Epoch 2/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 512ms/step - accuracy: 0.9090 - loss: 0.2314 - val_accuracy: 0.8815 - val_loss: 0.6970 - learning_rate: 9.9989e-05
Epoch 3/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 532ms/step - accuracy: 0.9458 - loss: 0.1384 - val_accuracy: 0.8773 - val_loss: 0.7867 - learning_rate: 9.9956e-05
Epoch 4/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 19s 541ms/step - accuracy: 0.9452 - loss: 0.1382 - val_accuracy: 0.8754 - val_loss: 0.8853 - learning_rate: 9.9901e-05
Epoch 5/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 527ms/step - accuracy: 0.9554 - loss: 0.1103 - val_accuracy: 0.8736 - val_loss: 0.9330 - learning_rate: 9.9825e-05
Epoch 6/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 518ms/step - accuracy: 0.9549 - loss: 0.1139 - val_accuracy: 0.8744 - val_loss: 0.9205 - learning_rate: 9.9726e-05
Epoch 7/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 483ms/step - accuracy: 0.

34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 539ms/step - accuracy: 0.9561 - loss: 0.1007 - val_accuracy: 0.8941 - val_loss: 0.6109 - learning_rate: 9.9606e-05
Epoch 8/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 489ms/step - accuracy: 0.9579 - loss: 0.1047

...

34/34 ━━━━━━━━━━━━━━━━━━━━ 19s 543ms/step - accuracy: 0.9706 - loss: 0.0726 - val_accuracy: 0.9693 - val_loss: 0.0622 - learning_rate: 8.7850e-05
Epoch 36/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 526ms/step - accuracy: 0.9708 - loss: 0.0702 - val_accuracy: 0.9686 - val_loss: 0.0630 - learning_rate: 8.7157e-05
Epoch 37/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 489ms/step - accuracy: 0.9693 - loss: 0.0750

34/34 ━━━━━━━━━━━━━━━━━━━━ 19s 543ms/step - accuracy: 0.9693 - loss: 0.0752 - val_accuracy: 0.9701 - val_loss: 0.0615 - learning_rate: 8.6448e-05
Epoch 38/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 523ms/step - accuracy: 0.9707 - loss: 0.0693 - val_accuracy: 0.9683 - val_loss: 0.0630 - learning_rate: 8.5724e-05
Epoch 39/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 487ms/step - accuracy: 0.9677 - loss: 0.0828

34/34 ━━━━━━━━━━━━━━━━━━━━ 19s 541ms/step - accuracy: 0.9677 - loss: 0.0828 - val_accuracy: 0.9698 - val_loss: 0.0613 - learning_rate: 8.4983e-05
Epoch 40/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 523ms/step - accuracy: 0.9699 - loss: 0.0714 - val_accuracy: 0.9653 - val_loss: 0.0682 - learning_rate: 8.4227e-05
Epoch 41/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 525ms/step - accuracy: 0.9694 - loss: 0.0769 - val_accuracy: 0.9656 - val_loss: 0.0681 - learning_rate: 8.3457e-05
Epoch 42/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 525ms/step - accuracy: 0.9710 - loss: 0.0766 - val_accuracy: 0.9687 - val_loss: 0.0631 - learning_rate: 8.2671e-05
Epoch 43/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 525ms/step - accuracy: 0.9700 - loss: 0.0718 - val_accuracy: 0.9689 - val_loss: 0.0620 - learning_rate: 8.1871e-05
Epoch 44/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 525ms/step - accuracy: 0.9715 - loss: 0.0690 - val_accuracy: 0.9655 - val_loss: 0.0682 - learning_rate: 8.1057e-05
Epoch 45/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 524ms/step - ac

34/34 ━━━━━━━━━━━━━━━━━━━━ 19s 543ms/step - accuracy: 0.9715 - loss: 0.0722 - val_accuracy: 0.9700 - val_loss: 0.0598 - learning_rate: 7.8536e-05
Epoch 48/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 487ms/step - accuracy: 0.9706 - loss: 0.0775

34/34 ━━━━━━━━━━━━━━━━━━━━ 19s 541ms/step - accuracy: 0.9706 - loss: 0.0775 - val_accuracy: 0.9701 - val_loss: 0.0596 - learning_rate: 7.7670e-05
Epoch 49/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 524ms/step - accuracy: 0.9706 - loss: 0.0695 - val_accuracy: 0.9643 - val_loss: 0.0699 - learning_rate: 7.6791e-05
Epoch 50/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 525ms/step - accuracy: 0.9720 - loss: 0.0663 - val_accuracy: 0.9682 - val_loss: 0.0628 - learning_rate: 7.5901e-05
Epoch 51/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 526ms/step - accuracy: 0.9709 - loss: 0.0708 - val_accuracy: 0.9633 - val_loss: 0.0715 - learning_rate: 7.5000e-05
Epoch 52/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 525ms/step - accuracy: 0.9690 - loss: 0.0773 - val_accuracy: 0.9701 - val_loss: 0.0598 - learning_rate: 7.4088e-05
Epoch 53/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 525ms/step - accuracy: 0.9730 - loss: 0.0655 - val_accuracy: 0.9699 - val_loss: 0.0597 - learning_rate: 7.3165e-05
Epoch 54/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 524ms/step - ac

In [ ]:
from google.colab import files

for fold in range(1, 6):
    files.download(f"r2unet_relu_t3_fold{fold}.h5")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

###Loss: Focal Tversky + Dice + BCE

In [ ]:
import zipfile, os, cv2, numpy as np, math
import tensorflow as tf
import albumentations as A
from sklearn.model_selection import KFold
from medpy.metric import binary


# Unzip Data
with zipfile.ZipFile('/content/stage1_train.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/stage1_train')
print("✅ Unzipped stage1_train.zip successfully!")

# Load Data
def load_dsb2018_data(dataset_dir, image_size=(256,256)):
    images, masks = [], []
    for folder in sorted(os.listdir(dataset_dir)):
        img_path = os.path.join(dataset_dir, folder, 'images', folder + '.png')
        mask_dir = os.path.join(dataset_dir, folder, 'masks')
        if not os.path.exists(img_path) or not os.path.exists(mask_dir):
            continue
        image = cv2.imread(img_path)
        image = cv2.resize(image, image_size).astype(np.float32)/255.0
        mask = np.zeros(image_size, dtype=np.uint8)
        for m in os.listdir(mask_dir):
            msk = cv2.imread(os.path.join(mask_dir, m), cv2.IMREAD_GRAYSCALE)
            msk = cv2.resize(msk, image_size)
            mask = np.maximum(mask, msk)
        mask = (mask>0).astype(np.float32)
        images.append(image)
        masks.append(np.expand_dims(mask, axis=-1))
    return np.array(images), np.array(masks)

X, y = load_dsb2018_data('/content/stage1_train', image_size=(256,256))

# =========================================================
# Augmentation
# =========================================================
transform = A.Compose([
    A.RandomCrop(width=224, height=224, p=1.0),    # random crop
    A.Resize(256, 256),                            # resize back to 256
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.GaussianBlur(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05,
                       rotate_limit=15, border_mode=cv2.BORDER_REFLECT_101, p=0.5),
    A.ElasticTransform(alpha=50, sigma=10, alpha_affine=10, p=0.3),  # elastic deformation
    A.GridDistortion(p=0.2),
    A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.2)
])

def augment(image, mask):
    augmented = transform(image=image, mask=mask)
    return augmented['image'], augmented['mask']

def tf_augment(img, mask):
    img, mask = tf.numpy_function(augment, [img, mask], [tf.float32, tf.float32])
    img.set_shape([256,256,3])
    mask.set_shape([256,256,1])
    return img, mask

# =========================================================
# Refined Loss Functions
# =========================================================
def tversky(y_true, y_pred, alpha=0.7, beta=0.3):
    smooth = 1e-6
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    tp = tf.reduce_sum(y_true * y_pred)
    fn = tf.reduce_sum(y_true * (1 - y_pred))
    fp = tf.reduce_sum((1 - y_true) * y_pred)
    return (tp + smooth) / (tp + alpha*fn + beta*fp + smooth)

def focal_tversky_loss(y_true, y_pred, gamma=1.0):
    tv = tversky(y_true, y_pred)
    return tf.pow((1 - tv), gamma)

def dice_loss(y_true, y_pred):
    smooth=1e-6
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1 - (2.*intersection + smooth)/(tf.reduce_sum(y_true_f)+tf.reduce_sum(y_pred_f)+smooth)

def bce_loss(y_true, y_pred):
    return tf.keras.losses.binary_crossentropy(y_true, y_pred)

def hybrid_loss(y_true, y_pred):
    return 0.5*focal_tversky_loss(y_true, y_pred) + 0.25*dice_loss(y_true, y_pred) + 0.25*bce_loss(y_true, y_pred)

# =========================================================
# R2U-Net Model
# =========================================================
class RecurrentConv(tf.keras.layers.Layer):
    def __init__(self, filters, t=2):
        super().__init__()
        self.filters = filters
        self.t = t
        self.activation = tf.keras.layers.Activation('relu')
        self.convs = [tf.keras.layers.Conv2D(filters, 3, padding='same') for _ in range(t)]
        self.bns = [tf.keras.layers.BatchNormalization() for _ in range(t)]
    def call(self, x):
        h = 0
        for i in range(self.t):
            h = self.activation(self.bns[i](self.convs[i](x + h))) if i>0 else self.activation(self.bns[i](self.convs[i](x)))
        return h

class RRU(tf.keras.layers.Layer):
    def __init__(self, filters, t=2):
        super().__init__()
        self.projection = tf.keras.layers.Conv2D(filters, 1, padding='same')
        self.rcl = RecurrentConv(filters, t)
    def call(self, x):
        x_proj = self.projection(x)
        return x_proj + self.rcl(x_proj)

def build_r2unet(input_shape=(256,256,3), num_classes=1, t=4):
    inputs = tf.keras.Input(shape=input_shape)
    # Encoder
    e1 = RRU(32, t)(inputs); p1=tf.keras.layers.MaxPooling2D()(e1)
    e2 = RRU(64, t)(p1); p2=tf.keras.layers.MaxPooling2D()(e2)
    e3 = RRU(128, t)(p2); p3=tf.keras.layers.MaxPooling2D()(e3)
    e4 = RRU(256, t)(p3); p4=tf.keras.layers.MaxPooling2D()(e4)
    # Bottleneck
    b = RRU(512, t)(p4)
    # Decoder
    u1 = tf.keras.layers.UpSampling2D()(b); u1=tf.keras.layers.Concatenate()([u1,e4]); d1 = RRU(256,t)(u1)
    u2 = tf.keras.layers.UpSampling2D()(d1); u2=tf.keras.layers.Concatenate()([u2,e3]); d2 = RRU(128,t)(u2)
    u3 = tf.keras.layers.UpSampling2D()(d2); u3=tf.keras.layers.Concatenate()([u3,e2]); d3 = RRU(64,t)(u3)
    u4 = tf.keras.layers.UpSampling2D()(d3); u4=tf.keras.layers.Concatenate()([u4,e1]); d4 = RRU(32,t)(u4)
    outputs = tf.keras.layers.Conv2D(num_classes,1,activation='sigmoid')(d4)
    return tf.keras.Model(inputs, outputs)

# =========================================================
# K-Fold Training
# =========================================================
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
all_fold_dice_scores = []

for train_idx, val_idx in kfold.split(X):
    print(f"========== Fold {fold} ==========")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    train_dataset = train_dataset.map(tf_augment, num_parallel_calls=tf.data.AUTOTUNE)
    train_dataset = train_dataset.shuffle(128).batch(16).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    val_dataset = val_dataset.batch(16).prefetch(tf.data.AUTOTUNE)

    lr_schedule = tf.keras.callbacks.LearningRateScheduler(lambda epoch: 1e-4*(1+math.cos(math.pi*epoch/150))/2)
    early_stop = tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)
    checkpoint = tf.keras.callbacks.ModelCheckpoint(f"best_r2unet_fold{fold}.h5", save_best_only=True)

    model = build_r2unet(input_shape=(256,256,3), t=4)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=hybrid_loss, metrics=['accuracy'])
    model.fit(train_dataset, validation_data=val_dataset, epochs=150, callbacks=[lr_schedule, early_stop, checkpoint], verbose=1)

    # Fold Evaluation
    dice_scores_fold = []
    preds = model.predict(X_val, batch_size=16, verbose=0)
    preds_bin = (preds>0.5).astype(np.uint8)
    for pb, gt in zip(preds_bin, y_val):
        if np.sum(pb)>0 and np.sum(gt)>0:
            dice_scores_fold.append(binary.dc(pb.squeeze(), gt.squeeze()))
    mean_dice_fold = np.mean(dice_scores_fold) if len(dice_scores_fold)>0 else 0
    print(f"✅ Fold {fold} Dice: {mean_dice_fold:.4f}")
    all_fold_dice_scores.append(mean_dice_fold)
    fold += 1

# =========================================================
# Final Average Dice
# =========================================================
print(f"✅ Average Dice across all folds: {np.mean(all_fold_dice_scores):.4f}")

✅ Unzipped stage1_train.zip successfully!


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipython-input-3014792500.py:46: UserWarning: Argument(s) 'alpha_affine' are not valid for transform ElasticTransform
  A.ElasticTransform(alpha=50, sigma=10, alpha_affine=10, p=0.3),  # elastic deformation
/tmp/ipython-input-3014792500.py:48: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.2)


========== Fold 1 ==========
Epoch 1/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7956 - loss: 0.4923   

34/34 ━━━━━━━━━━━━━━━━━━━━ 108s 2s/step - accuracy: 0.7979 - loss: 0.4888 - val_accuracy: 0.8475 - val_loss: 0.6567 - learning_rate: 1.0000e-04
Epoch 2/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 693ms/step - accuracy: 0.9325 - loss: 0.2452 - val_accuracy: 0.8722 - val_loss: 0.7638 - learning_rate: 9.9989e-05
Epoch 3/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 25s 723ms/step - accuracy: 0.9421 - loss: 0.2088 - val_accuracy: 0.8729 - val_loss: 0.8514 - learning_rate: 9.9956e-05
Epoch 4/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 699ms/step - accuracy: 0.9460 - loss: 0.1925 - val_accuracy: 0.8728 - val_loss: 0.9260 - learning_rate: 9.9901e-05
Epoch 5/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 694ms/step - accuracy: 0.9513 - loss: 0.1739 - val_accuracy: 0.8758 - val_loss: 0.8615 - learning_rate: 9.9825e-05
Epoch 6/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 706ms/step - accuracy: 0.9460 - loss: 0.1929 - val_accuracy: 0.8800 - val_loss: 0.7966 - learning_rate: 9.9726e-05
Epoch 7/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 660ms/step - accuracy: 0

34/34 ━━━━━━━━━━━━━━━━━━━━ 25s 730ms/step - accuracy: 0.9519 - loss: 0.1728 - val_accuracy: 0.8959 - val_loss: 0.6360 - learning_rate: 9.9606e-05
Epoch 8/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 656ms/step - accuracy: 0.9536 - loss: 0.1603

...

34/34 ━━━━━━━━━━━━━━━━━━━━ 25s 732ms/step - accuracy: 0.9622 - loss: 0.1478 - val_accuracy: 0.9605 - val_loss: 0.1296 - learning_rate: 9.6489e-05
Epoch 20/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 659ms/step - accuracy: 0.9612 - loss: 0.1492

34/34 ━━━━━━━━━━━━━━━━━━━━ 25s 731ms/step - accuracy: 0.9612 - loss: 0.1493 - val_accuracy: 0.9648 - val_loss: 0.1195 - learning_rate: 9.6093e-05
Epoch 21/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 660ms/step - accuracy: 0.9649 - loss: 0.1244

34/34 ━━━━━━━━━━━━━━━━━━━━ 25s 732ms/step - accuracy: 0.9649 - loss: 0.1245 - val_accuracy: 0.9634 - val_loss: 0.1178 - learning_rate: 9.5677e-05
Epoch 22/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 706ms/step - accuracy: 0.9621 - loss: 0.1392 - val_accuracy: 0.9520 - val_loss: 0.1363 - learning_rate: 9.5241e-05
Epoch 23/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 660ms/step - accuracy: 0.9633 - loss: 0.1345

34/34 ━━━━━━━━━━━━━━━━━━━━ 25s 732ms/step - accuracy: 0.9633 - loss: 0.1345 - val_accuracy: 0.9668 - val_loss: 0.1095 - learning_rate: 9.4786e-05
Epoch 24/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 40s 710ms/step - accuracy: 0.9643 - loss: 0.1253 - val_accuracy: 0.9689 - val_loss: 0.1126 - learning_rate: 9.4310e-05
Epoch 25/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 25s 732ms/step - accuracy: 0.9615 - loss: 0.1435 - val_accuracy: 0.9627 - val_loss: 0.1151 - learning_rate: 9.3815e-05
Epoch 26/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 698ms/step - accuracy: 0.9637 - loss: 0.1307 - val_accuracy: 0.9641 - val_loss: 0.1139 - learning_rate: 9.3301e-05
Epoch 27/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 696ms/step - accuracy: 0.9609 - loss: 0.1438 - val_accuracy: 0.9602 - val_loss: 0.1226 - learning_rate: 9.2768e-05
Epoch 28/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 709ms/step - accuracy: 0.9627 - loss: 0.1335 - val_accuracy: 0.9565 - val_loss: 0.1285 - learning_rate: 9.2216e-05
Epoch 29/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 662ms/step - acc

34/34 ━━━━━━━━━━━━━━━━━━━━ 25s 734ms/step - accuracy: 0.9619 - loss: 0.1358 - val_accuracy: 0.9680 - val_loss: 0.1059 - learning_rate: 9.1646e-05
Epoch 30/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 699ms/step - accuracy: 0.9650 - loss: 0.1279 - val_accuracy: 0.9691 - val_loss: 0.1069 - learning_rate: 9.1057e-05
Epoch 31/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 703ms/step - accuracy: 0.9625 - loss: 0.1388 - val_accuracy: 0.9646 - val_loss: 0.1122 - learning_rate: 9.0451e-05
Epoch 32/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 709ms/step - accuracy: 0.9652 - loss: 0.1274 - val_accuracy: 0.9680 - val_loss: 0.1156 - learning_rate: 8.9826e-05
Epoch 33/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 704ms/step - accuracy: 0.9651 - loss: 0.1279 - val_accuracy: 0.9640 - val_loss: 0.1099 - learning_rate: 8.9185e-05
Epoch 34/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 707ms/step - accuracy: 0.9607 - loss: 0.1408 - val_accuracy: 0.9679 - val_loss: 0.1077 - learning_rate: 8.8526e-05
Epoch 35/150
34/34 ━━━━━━━━━━━━━━━━━━━━ 24s 707ms/step - ac

In [ ]:
from google.colab import files

for fold in range(1, 6):
    files.download(f"best_r2unet_fold{fold}.h5")